# Amazon-BuyBox-Thesis-Figures

Secondary notebook to the Master's thesis *Amazon-BuyBox-Thesis-Econometrics-Analysis-Notebook*. It rebuilds the final third-party seller-market panel from the
audited Xiaomi Mi Smart Band 6 raw extraction and renders the figures
included in `main.tex` via `\includegraphics`.

The reconstructed panel matches the audited sample of 5,107
seller-market rows across 62 timestamped markets (996 FBA rows, 4,111
non-FBA rows). Each figure is written without an in-figure title or
in-figure global caption, so that the LaTeX caption typeset by the
thesis template is the only descriptive layer. Figure-by-figure
provenance, label, and placement in `main.tex` are reported in the
corresponding markdown cell below.


## Software environment

The Drive mount and the core-package check below replicate the analysis
notebook, so the figure pipeline runs in the same Colab environment.
Package versions are not pinned because the analysis notebook also runs
on the default Colab kernel and the numerical results in the thesis are
not version-sensitive at the precision reported in the tables.


In [ ]:
# -----------------------------------------------------------------------------
# Google Drive mount and core-package availability check.
# Mounts Drive on Colab, verifies that the core scientific stack is
# importable, and prints any missing package before the figure
# pipeline starts.
# -----------------------------------------------------------------------------

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is unavailable; continuing with local file resolution.')
except Exception as exc:
    print(f'Google Drive mount skipped: {exc}')

_required_packages = ['numpy', 'pandas', 'statsmodels', 'scipy', 'patsy', 'matplotlib']
_missing_packages = []
for _pkg in _required_packages:
    try:
        __import__(_pkg)
    except ModuleNotFoundError:
        _missing_packages.append(_pkg)
if _missing_packages:
    print('Missing packages:', _missing_packages)
    print('Install them before running the full notebook, for example with %pip install statsmodels scipy patsy matplotlib')
else:
    print('Required core packages are importable.')


## Imports, output paths, and matplotlib configuration

The figures are written to a local staging directory first and then
mirrored to
`Amazon-BuyBox-Econometrics-Analysis/Datasets/Figures` on Drive when
Drive is mounted. The local-then-mirror pattern, together with bounded
retries on `OSError`, isolates the matplotlib PDF backend from the
transient Drive FUSE error
`OSError: [Errno 107] Transport endpoint is not connected`, which is
otherwise raised intermittently when many incremental PDF writes hit
the mounted Drive in quick succession.


In [ ]:
# -----------------------------------------------------------------------------
# Imports and figure-output configuration.
# -----------------------------------------------------------------------------
import os
import re
import math
import json
import unicodedata
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import linalg, stats
from patsy import dmatrices
from statsmodels.stats.sandwich_covariance import cov_cluster_2groups

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 160)

NOTEBOOK_VERSION = 'figures-pilot-2026-05-20'

# Typographic configuration. The thesis is composed in Helvetica sans-serif
# (\usepackage{helvet} with \familydefault=\sfdefault under pdfLaTeX, or
# TeX Gyre Heros under XeLaTeX or LuaLaTeX. The figure face matches the body face. The sans-serif
# fallback list is ordered by preference; Colab usually ships with at
# least one of Helvetica, TeX Gyre Heros, or Nimbus Sans. DejaVu Sans is
# the visually adjacent last-resort fallback.
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'TeX Gyre Heros', 'Nimbus Sans',
                        'Arial', 'Liberation Sans', 'DejaVu Sans'],
    'mathtext.fontset': 'stixsans',
    'axes.unicode_minus': False,
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.figsize': (5.5, 3.5),
    'figure.dpi': 110,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'lines.linewidth': 1.4,
    'patch.linewidth': 0.8,
    'errorbar.capsize': 3.0,
})

# Output directories.
#
# The Drive FUSE layer on Colab occasionally raises
#   OSError: [Errno 107] Transport endpoint is not connected
# when matplotlib's PDF backend issues many small incremental writes to a
# path on the mounted Drive (PNG and PDF saved in rapid sequence to a
# Drive directory). The figure is therefore written to a local staging
# directory first, then mirrored to the Drive directory with bounded
# retries on the transient OSError. The local copy is also kept on disk
# so reruns are immediate.
import shutil
import time

LOCAL_STAGING_DIR = (Path.cwd() / 'figures_local').resolve()
LOCAL_STAGING_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_DATASETS_DIR = Path('/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets')
if DRIVE_DATASETS_DIR.exists():
    FIGURES_DIR = DRIVE_DATASETS_DIR / 'Figures'
else:
    FIGURES_DIR = LOCAL_STAGING_DIR
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def _robust_copy(src_path, dst_path, max_attempts=5, base_delay=0.5):
    # Copy a file with bounded retries on the transient Drive FUSE OSError.
    last_exc = None
    for attempt in range(max_attempts):
        try:
            shutil.copyfile(src_path, dst_path)
            return dst_path
        except OSError as exc:
            last_exc = exc
            time.sleep(base_delay * (2 ** attempt))
    raise last_exc


def save_figure(fig, stem):
    # Persist a figure to the local staging directory, then mirror to
    # FIGURES_DIR. matplotlib always writes to the local-disk staging
    # path; if Drive is the final target, a separate shutil.copyfile
    # step (with retries) moves the finished file. This isolates the
    # matplotlib backend from Drive FUSE transients and reduces the
    # number of independent write operations on Drive.
    for ext in ('png', 'pdf'):
        staging_path = LOCAL_STAGING_DIR / f'{stem}.{ext}'
        fig.savefig(staging_path)
        if FIGURES_DIR.resolve() != LOCAL_STAGING_DIR:
            _robust_copy(staging_path, FIGURES_DIR / f'{stem}.{ext}')
    plt.close(fig)
    return FIGURES_DIR / f'{stem}.png'


print('Local staging directory:', LOCAL_STAGING_DIR)
print('Final figures directory:', FIGURES_DIR)

# Shared palette used across F1, F6, F11, F12 and several appendix
# figures. FBA is tied to a warm orange, non-FBA to a cool blue, each in
# two intensities: a darker tone for line outlines and a lighter tone for
# fills. Common-support regions use a neutral light grey.
COLOR_FBA_LINE    = '#C97E3A'     # darker orange for curve lines and markers
COLOR_FBA_FILL    = '#F4B47A'     # lighter orange for area fills
COLOR_NONFBA_LINE = '#3D7BAB'     # darker blue for curve lines and markers
COLOR_NONFBA_FILL = '#9CC4E0'     # lighter blue for area fills
COLOR_COMMON_SUPPORT = '#D9D9D9'  # neutral grey for out-of-support shading
ALPHA_FILL = 0.55
ALPHA_COMMON_SUPPORT = 0.55


def _density_legend_handles(n_fba, n_non, include_median=False):
    # Composite legend handles: each entry combines a coloured line and a
    # coloured patch so that the curve and the filled area appear together
    # in the legend symbol.
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    handles = [
        (Line2D([0], [0], color=COLOR_FBA_LINE,    linewidth=1.6, linestyle='-'),
         Patch(facecolor=COLOR_FBA_FILL,    alpha=ALPHA_FILL, edgecolor='none')),
        (Line2D([0], [0], color=COLOR_NONFBA_LINE, linewidth=1.6, linestyle='--'),
         Patch(facecolor=COLOR_NONFBA_FILL, alpha=ALPHA_FILL, edgecolor='none')),
    ]
    labels = [f'FBA (n={n_fba:,})', f'non-FBA (n={n_non:,})']
    if include_median:
        handles.append(Line2D([0], [0], color='black', linestyle='-',
                              linewidth=0.8, alpha=0.6))
        labels.append('per-group median')
    return handles, labels


## Input file resolution

The candidate-path list replicates the analysis notebook. The Drive
location is the last-resort search path.


In [ ]:
# -----------------------------------------------------------------------------
# Path resolver for the raw Xiaomi Mi Smart Band 6 CSV.
# -----------------------------------------------------------------------------
DATA_PATH_OVERRIDE = None

CANDIDATE_FILE_NAMES = [
    'Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6 - Foglio1.csv',
    'Sport e tempo libero - Smartwatch - Xiaomi Mi Smart Band 6.csv',
]

CANDIDATE_DIRS = [
    Path.cwd(),
    Path('/mnt/data'),
    Path('.'),
    Path('/content'),
    Path('/content/drive/MyDrive/Amazon-BuyBox-Econometrics-Analysis/Datasets'),
]


def resolve_input_file(data_path_override=None, candidate_names=None, candidate_dirs=None):
    if data_path_override is not None:
        p = Path(data_path_override)
        if p.exists():
            return p.resolve()
        raise FileNotFoundError(f'Explicit DATA_PATH_OVERRIDE not found: {data_path_override}')

    candidate_names = candidate_names or CANDIDATE_FILE_NAMES
    candidate_dirs = candidate_dirs or CANDIDATE_DIRS

    for directory in candidate_dirs:
        for name in candidate_names:
            p = directory / name
            if p.exists():
                return p.resolve()

    search_roots = [Path.cwd(), Path('/mnt/data'), Path('/content')]
    for root in search_roots:
        if root.exists():
            for name in candidate_names:
                hits = list(root.rglob(name))
                if hits:
                    return hits[0].resolve()

    raise FileNotFoundError(
        'Xiaomi CSV not found in any expected location. '
        'Set DATA_PATH_OVERRIDE explicitly if needed.'
    )


file_path = resolve_input_file(DATA_PATH_OVERRIDE)
print('Resolved input path:', file_path)


## Parsing helpers

The string, numeric, shipping, and delivery parsers are reproduced
verbatim from the analysis notebook. Reproducing the parsers verbatim
is what allows the reconstructed panel to reconcile exactly with the
audited 5,107-row sample (Chapter `\ref{ch:data}`, Section
`\ref{sec:data-audit}` of the thesis).


In [ ]:
# -----------------------------------------------------------------------------
# String and numeric parsers. Reproduced verbatim from the analysis
# notebook so that the rebuilt panel reconciles to the audited counts.
# -----------------------------------------------------------------------------
def strip_accents(text):
    return ''.join(ch for ch in unicodedata.normalize('NFKD', text)
                   if not unicodedata.combining(ch))


def normalize_text(value):
    if value is None or pd.isna(value):
        return ''
    s = str(value).strip().lower()
    s = strip_accents(s)
    s = re.sub(r'\s+', ' ', s)
    return s


def slugify_text(value):
    return re.sub(r'[^a-z0-9]+', '_', normalize_text(value)).strip('_')


def third_party_from_name(seller_name):
    return not bool(re.search(r'\bamazon\b', normalize_text(seller_name)))


def fba_from_shipper(shipper_name):
    return bool(re.search(r'\bamazon\b', normalize_text(shipper_name)))


def parse_numeric_italian(series):
    s = series.astype('string')
    s = s.str.replace('.', '', regex=False)
    s = s.str.replace(',', '.', regex=False)
    return pd.to_numeric(s, errors='coerce')


def parse_boolean_flag(series):
    s = series.astype('string').str.strip().str.lower()
    return s.map({'true': True, 'false': False, '1': True, '0': False}).astype('boolean')


def clean_string_columns(df):
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            s = df[col].astype('string').str.strip()
            s = s.replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
            df[col] = s
    return df


# -----------------------------------------------------------------------------
# Shipping and delivery text parsing. Italian-language patterns from the
# displayed-shipping audit of Chapter \ref{ch:data}.
# -----------------------------------------------------------------------------
MONTH_MAP = {
    'gen': 1, 'gennaio': 1, 'feb': 2, 'febbraio': 2, 'mar': 3, 'marzo': 3,
    'apr': 4, 'aprile': 4, 'mag': 5, 'maggio': 5, 'giu': 6, 'giugno': 6,
    'lug': 7, 'luglio': 7, 'ago': 8, 'agosto': 8,
    'set': 9, 'sett': 9, 'settembre': 9,
    'ott': 10, 'ottobre': 10, 'nov': 11, 'novembre': 11, 'dic': 12, 'dicembre': 12,
}

range_two_months_re = re.compile(r'(\d{1,2})\s+([a-z]+)\s*-\s*(\d{1,2})\s+([a-z]+)')
range_same_month_re = re.compile(r'(\d{1,2})\s*-\s*(\d{1,2})\s+([a-z]+)')
single_date_re = re.compile(
    r'(?:lunedi|martedi|mercoledi|giovedi|venerdi|sabato|domenica)?\s*,?\s*(\d{1,2})\s+([a-z]+)'
)

robust_fast_split_re = re.compile(r'(?:oppure\s+)?consegna\s+piu\s+(?:veloce|rapida)\s*:?', re.I)
robust_primary_prefix_re = re.compile(
    r'^(?:spedizione\s+gratuita\s*:|consegna\s+gratuita\s*|'
    r'consegna\s+a\s*[0-9]+(?:[\.,][0-9]+)?\s*\u20ac?\s*:?)', re.I,
)
shipping_price_re = re.compile(r'consegna\s+a\s*([0-9]+(?:[\.,][0-9]+)?)\s*\u20ac', re.I)
shipping_free_re = re.compile(r'(spedizione|consegna)\s+gratuita', re.I)
contact_courier_re = re.compile(r'verrai\s+contattato\s+dal\s+corriere', re.I)


def normalize_delivery_text(text):
    if text is None or pd.isna(text):
        return None
    s = str(text).lower()
    s = strip_accents(s)
    s = s.replace('maggiori informazioni', ' ')
    s = re.sub(r'ordina entro [^.]*', ' ', s)
    s = s.replace('sul tuo primo ordine idoneo', ' ')
    s = s.replace('.', ' ')
    s = s.replace(':', ' : ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def build_delivery_date(ts, day, month_token):
    month_token = month_token.strip().lower()
    month = MONTH_MAP.get(month_token, MONTH_MAP.get(month_token[:3]))
    if month is None:
        return None
    year = ts.year
    if month < ts.month - 1:
        year += 1
    try:
        return datetime(year, month, int(day)).date()
    except ValueError:
        return None


def parse_delivery_window(segment, ts):
    if segment is None or pd.isna(segment) or pd.isna(ts):
        return (np.nan, np.nan)
    s = segment.strip()
    m = range_two_months_re.search(s)
    if m:
        d1, m1, d2, m2 = m.groups()
        dt1 = build_delivery_date(ts, d1, m1)
        dt2 = build_delivery_date(ts, d2, m2)
        if dt1 is not None and dt2 is not None:
            return ((dt1 - ts.date()).days, (dt2 - ts.date()).days)
    m = range_same_month_re.search(s)
    if m:
        d1, d2, m1 = m.groups()
        dt1 = build_delivery_date(ts, d1, m1)
        dt2 = build_delivery_date(ts, d2, m1)
        if dt1 is not None and dt2 is not None:
            return ((dt1 - ts.date()).days, (dt2 - ts.date()).days)
    m = single_date_re.search(s)
    if m:
        d1, m1 = m.groups()
        dt = build_delivery_date(ts, d1, m1)
        if dt is not None:
            diff = (dt - ts.date()).days
            return (diff, diff)
    return (np.nan, np.nan)


def parse_shipping_from_text(text):
    # Match order: explicit price, then contact-courier, then explicit
    # free shipping. Unrecognised text returns pd.NA rather than a
    # placeholder mode.
    norm = normalize_delivery_text(text)
    if norm is None:
        return {'parsed_shipping_mode': pd.NA, 'parsed_shipping_price': np.nan}
    m = shipping_price_re.search(norm)
    if m:
        return {'parsed_shipping_mode': 'paid_explicit',
                'parsed_shipping_price': float(m.group(1).replace(',', '.'))}
    if contact_courier_re.search(norm):
        return {'parsed_shipping_mode': 'unknown_contact_courier',
                'parsed_shipping_price': np.nan}
    if shipping_free_re.search(norm):
        return {'parsed_shipping_mode': 'free_explicit',
                'parsed_shipping_price': 0.0}
    return {'parsed_shipping_mode': pd.NA, 'parsed_shipping_price': np.nan}


def parse_delivery_robust(text, ts):
    s = normalize_delivery_text(text)
    if s is None or pd.isna(ts):
        return [np.nan, np.nan, np.nan, np.nan]
    parts = robust_fast_split_re.split(s, maxsplit=1)
    primary = robust_primary_prefix_re.sub('', parts[0]).strip()
    fast = parts[1].strip() if len(parts) > 1 else None
    g_min, g_max = parse_delivery_window(primary, ts)
    gv_min, gv_max = parse_delivery_window(fast, ts) if fast else (0, 0)
    if pd.isna(g_min) and not pd.isna(gv_min):
        g_min, g_max, gv_min, gv_max = gv_min, gv_max, 0, 0
    return [g_min, g_max, gv_min, gv_max]


## Reconstruction of the final third-party panel

The construction follows the analysis notebook step by step (sample
restrictions; shipping and delivery audited repairs; reconstructed total
buyer-facing price; within-market normalised rank `rank_pct`;
chronological market index used by Figure F12). The assertion at the
end of the cell reconciles the reconstructed panel against the audited
row counts: 5,107 rows across 62 markets, 996 FBA rows and 4,111
non-FBA rows, 50 contact-courier rows. These are the counts reported in
Table `\ref{tab:rank-summary-fba}` of Chapter `\ref{ch:data}`.


In [ ]:
# -----------------------------------------------------------------------------
# Final third-party seller-market panel.
# Reconciles to 5,107 rows across 62 markets (996 FBA, 4,111 non-FBA,
# 50 contact-courier rows), as reported in Chapter \ref{ch:data} and
# in Table \ref{tab:rank-summary-fba}.
# -----------------------------------------------------------------------------
PRODUCT_LABEL = 'Xiaomi Mi Smart Band 6'

raw_df = pd.read_csv(file_path).copy()
raw_df.insert(0, 'raw_row_id', np.arange(len(raw_df), dtype=int))
raw_df['product_name'] = PRODUCT_LABEL
raw_df = clean_string_columns(raw_df)

raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], errors='coerce')
raw_df['timestamp_date'] = raw_df['timestamp'].dt.normalize()

raw_df['seller_name'] = raw_df['venduto_da'].astype('string').str.strip()
raw_df['shipper_name'] = raw_df['spedito_da'].astype('string').str.strip()
raw_df['seller_name_norm'] = raw_df['seller_name'].map(normalize_text)
raw_df['seller_id'] = raw_df['seller_name'].map(slugify_text)
raw_df['shipper_name_norm'] = raw_df['shipper_name'].map(normalize_text)

raw_df['raw_identity_problem_flag'] = raw_df['seller_name'].isna() | raw_df['shipper_name'].isna()
raw_df['condizione'] = raw_df['condizione'].astype('string').str.strip()
raw_df['third_party_from_name'] = raw_df['seller_name'].map(third_party_from_name)
raw_df['fba_from_shipper'] = raw_df['shipper_name'].map(fba_from_shipper)
raw_df['fba_raw'] = parse_boolean_flag(raw_df['fba'])

numeric_map = {
    'prezzo': 'prezzo',
    'prezzo_prod_venduto(\u20ac)': 'prezzo_prodotto_venduto',
    'prezzo_spedizione(\u20ac)': 'prezzo_spedizione_raw',
    'prezzo_totale(\u20ac)': 'prezzo_totale_raw',
    'stelle': 'stelle',
}
for old_col, new_col in numeric_map.items():
    if old_col in raw_df.columns:
        raw_df[new_col] = parse_numeric_italian(raw_df[old_col])

integer_columns = [
    'buy_box', 'visibility_order', 'num_valutazioni', 'valutazioni_positive',
    'qta_min', 'g_cons_min', 'g_cons_max', 'g_spedizione',
    'g_cons_vel_min', 'g_cons_vel_max', 'g_spedizione_vel',
]
for col in integer_columns:
    if col in raw_df.columns:
        raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')

preferred_offer_sample = (
    raw_df.loc[
        raw_df['condizione'].eq('Nuovo')
        & raw_df['third_party_from_name']
        & (~raw_df['raw_identity_problem_flag'])
    ]
    .copy()
    .sort_values(['timestamp', 'seller_name_norm', 'visibility_order', 'raw_row_id'],
                 kind='mergesort')
)

df_final = (
    preferred_offer_sample.groupby(['timestamp', 'seller_name_norm'],
                                   as_index=False, sort=False)
    .first()
    .copy()
)

df_final['entity_time_id'] = (
    df_final['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S') + '__' + df_final['seller_id']
)

# Shipping reconstruction from displayed-text audit.
shipping_text_parsed = df_final['spedizione_consegna'].apply(parse_shipping_from_text).apply(pd.Series)
df_final = pd.concat([df_final, shipping_text_parsed], axis=1)
df_final['shipping_amount_text'] = df_final['parsed_shipping_price']
df_final['shipping_mode_text'] = df_final['parsed_shipping_mode']
df_final['contact_courier_flag'] = df_final['shipping_mode_text'].eq('unknown_contact_courier')

df_final['prezzo_spedizione_repaired'] = np.where(
    df_final['shipping_amount_text'].notna(),
    df_final['shipping_amount_text'],
    df_final['prezzo_spedizione_raw'],
)

# Delivery reconstruction from displayed-text audit.
delivery_parsed = df_final.apply(
    lambda row: pd.Series(
        parse_delivery_robust(row['spedizione_consegna'], row['timestamp']),
        index=['parsed_g_cons_min', 'parsed_g_cons_max',
               'parsed_g_cons_vel_min', 'parsed_g_cons_vel_max'],
    ),
    axis=1,
)
df_final = pd.concat([df_final, delivery_parsed], axis=1)

market_delivery_flags = pd.concat(
    [
        df_final.groupby('timestamp')['g_cons_min']
                .apply(lambda s: s.fillna(0).eq(0).all())
                .rename('market_delivery_all_zero_flag'),
        df_final.groupby('timestamp')['parsed_g_cons_min']
                .apply(lambda s: s.fillna(0).gt(0).any())
                .rename('market_delivery_text_support_flag'),
    ],
    axis=1,
)
market_delivery_flags['market_delivery_failure_flag'] = (
    market_delivery_flags['market_delivery_all_zero_flag']
    & market_delivery_flags['market_delivery_text_support_flag']
)
df_final = df_final.merge(
    market_delivery_flags[['market_delivery_failure_flag']],
    on='timestamp', how='left',
)

df_final['delivery_repaired_flag'] = (
    df_final['market_delivery_failure_flag'] & df_final['parsed_g_cons_min'].notna()
)
df_final['g_cons_min_robust'] = np.where(
    df_final['delivery_repaired_flag'],
    df_final['parsed_g_cons_min'],
    df_final['g_cons_min'],
)

# Reconstructed total buyer-facing price and reputation derived variables.
df_final['prezzo_totale_reconstructed'] = (
    df_final['prezzo'] + df_final['prezzo_spedizione_repaired']
)
df_final['review_support_positive_flag'] = (
    df_final['num_valutazioni'].fillna(0).gt(0)
    & df_final['valutazioni_positive'].fillna(0).gt(0)
    & df_final['stelle'].fillna(0).gt(0)
)
df_final['review_problem_flag'] = ~df_final['review_support_positive_flag']
df_final['log1p_num_valutazioni'] = np.log1p(df_final['num_valutazioni'].clip(lower=0))

# Within-market ordering and normalized rank.
df_final = df_final.sort_values(
    ['timestamp', 'visibility_order', 'seller_name_norm', 'raw_row_id'],
    kind='mergesort',
).copy()
df_final['market_id'] = df_final['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S')
df_final['market_n_sellers'] = df_final.groupby('market_id')['entity_time_id'].transform('size')
df_final['rank_pos'] = df_final.groupby('market_id').cumcount() + 1
df_final['rank_pct'] = np.where(
    df_final['market_n_sellers'].gt(1),
    (df_final['rank_pos'] - 1) / (df_final['market_n_sellers'] - 1),
    0.0,
)

# Chronological market index, used by F12 and by the per-market
# diagnostics F14, FC, FF.
market_order = (
    pd.Series(sorted(df_final['market_id'].unique()), name='market_id')
    .to_frame()
    .assign(market_order=lambda d: np.arange(1, len(d) + 1, dtype=int))
)
df_final = df_final.merge(market_order, on='market_id', how='left')

for col in ['fba_from_shipper', 'contact_courier_flag', 'review_problem_flag',
            'market_delivery_failure_flag']:
    df_final[col] = df_final[col].astype(int)

# Audit reconciliation against the audited counts reported in the thesis.
EXPECTED_AUDIT_COUNTS = {
    'final_rows': 5107, 'n_markets': 62,
    'fba_rows': 996, 'non_fba_rows': 4111,
    'contact_courier_rows': 50,
}
actual_counts = {
    'final_rows': int(len(df_final)),
    'n_markets': int(df_final['market_id'].nunique()),
    'fba_rows': int(df_final['fba_from_shipper'].sum()),
    'non_fba_rows': int((1 - df_final['fba_from_shipper']).sum()),
    'contact_courier_rows': int(df_final['contact_courier_flag'].sum()),
}
audit = pd.DataFrame({
    'expected': pd.Series(EXPECTED_AUDIT_COUNTS),
    'actual':   pd.Series(actual_counts),
})
audit['matches'] = audit['expected'].eq(audit['actual'])
print(audit)
assert audit['matches'].all(), 'Reconstructed panel does not match the audited counts.'


## OLS with two-way clustered inference

The static specifications S1 to S4 use within-market projections with
two-way seller-and-market clustered standard errors (Chapter
`\ref{app:econometric-spec}` of the thesis appendix). The helpers below
reproduce the same estimator used by the analysis notebook; they are
used in the figures cell only for the Frisch--Waugh--Lovell
constructions (F3, F19) and for the attenuation accounting (FA, FB) so
that the reported slopes coincide with the table coefficients.


In [ ]:
# -----------------------------------------------------------------------------
# Static specifications S1 to S4 and clustered-inference helpers.
# Subset of the analysis-notebook estimator used by the FWL constructions
# (F3, F19) and by the attenuation accounting (FA, FB).
# -----------------------------------------------------------------------------
CONTINUOUS_OUTCOME = 'rank_pct'
HEADLINE_SPEC = 'spec_4_fba_reputation_price_shipping_delivery'

SPECIFICATIONS = {
    'spec_1_fba_only': ['fba_from_shipper'],
    'spec_2_fba_reputation': [
        'fba_from_shipper', 'log1p_num_valutazioni',
        'valutazioni_positive', 'stelle',
    ],
    'spec_3_fba_reputation_totalprice_delivery': [
        'fba_from_shipper', 'log1p_num_valutazioni',
        'valutazioni_positive', 'stelle',
        'prezzo_totale_reconstructed', 'contact_courier_flag',
        'g_cons_min_robust',
    ],
    'spec_4_fba_reputation_price_shipping_delivery': [
        'fba_from_shipper', 'log1p_num_valutazioni',
        'valutazioni_positive', 'stelle',
        'prezzo', 'prezzo_spedizione_repaired',
        'contact_courier_flag', 'g_cons_min_robust',
    ],
}


def model_formula(rhs, outcome=CONTINUOUS_OUTCOME, include_market_fe=True):
    terms = list(rhs)
    if include_market_fe:
        terms = terms + ['C(market_id)']
    return outcome + ' ~ ' + ' + '.join(terms)


def cluster_codes(series):
    return pd.Categorical(series).codes


def stable_symmetric_pinv(matrix, ridge_floor=1e-10, max_tries=8):
    A = np.asarray(matrix, dtype=float)
    A = (A + A.T) / 2.0
    k = A.shape[0]
    if k == 0:
        return A.copy()
    eye = np.eye(k)
    scale = max(abs(float(np.trace(A) / k)), 1.0)
    try:
        return linalg.solve(A, eye, assume_a='sym', check_finite=False)
    except Exception:
        pass
    for j in range(max_tries):
        ridge = ridge_floor * (10 ** j) * scale
        try:
            return linalg.solve(A + ridge * eye, eye, assume_a='pos', check_finite=False)
        except Exception:
            continue
    return linalg.pinv(A, rtol=1e-10, check_finite=False)


def one_way_cluster_cov(X, resid, groups, bread):
    X = np.asarray(X, dtype=float)
    resid = np.asarray(resid, dtype=float)
    groups = np.asarray(groups)
    n, p = X.shape
    unique_groups, inverse = np.unique(groups, return_inverse=True)
    G = len(unique_groups)
    if G == n:
        meat = X.T @ (X * (resid ** 2)[:, None])
    else:
        scores = np.zeros((G, p), dtype=float)
        np.add.at(scores, inverse, X * resid[:, None])
        meat = scores.T @ scores
    correction = (G / (G - 1)) * ((n - 1) / (n - p)) if G > 1 and n > p else 1.0
    return correction * (bread @ meat @ bread)


def two_way_cluster_cov(X, resid, seller_groups, market_groups, bread):
    seller_cov = one_way_cluster_cov(X, resid, seller_groups, bread)
    market_cov = one_way_cluster_cov(X, resid, market_groups, bread)
    inter = pd.Categorical(pd.Series(seller_groups).astype(str)
                           + '__' + pd.Series(market_groups).astype(str)).codes
    inter_cov = one_way_cluster_cov(X, resid, inter, bread)
    return seller_cov + market_cov - inter_cov


def estimate_with_two_way_clustering(data, rhs):
    # Return the FBA point estimate, two-way clustered SE, t-statistic,
    # p-value, and the cluster-t 95% confidence interval.
    formula = model_formula(rhs)
    y_df, x_df = dmatrices(formula, data=data, return_type='dataframe')
    y = np.asarray(y_df).ravel()
    X = np.asarray(x_df, dtype=float)
    names = list(x_df.columns)
    bread = stable_symmetric_pinv(X.T @ X)
    beta = bread @ X.T @ y
    resid = y - X @ beta
    design_data = data.loc[x_df.index]
    seller = cluster_codes(design_data['seller_id'])
    market = cluster_codes(design_data['market_id'])
    cov = two_way_cluster_cov(X, resid, seller, market, bread)
    fba_idx = names.index('fba_from_shipper')
    n_clusters = max(min(pd.Series(seller).nunique(),
                         pd.Series(market).nunique()) - 1, 1)
    se = float(np.sqrt(max(cov[fba_idx, fba_idx], 0.0)))
    coef = float(beta[fba_idx])
    tstat = coef / se if se > 0 else np.nan
    pval = float(2 * (1 - stats.t.cdf(abs(tstat), df=n_clusters))) if np.isfinite(tstat) else np.nan
    crit = float(stats.t.ppf(0.975, df=n_clusters))
    return {
        'coef': coef, 'se': se, 't': tstat, 'p': pval,
        'ci_low': coef - crit * se, 'ci_high': coef + crit * se,
        'n_obs': int(len(y)),
    }


---

## Figure F1. Normalized within-market rank distribution by FBA status

LaTeX label: `fig:rank-pct-density`. File:
`fig_F1_rank_pct_density.pdf`. Rendered in Chapter
`\ref{ch:data}`, Section `\ref{sec:data-descriptive}` of `main.tex`,
between Table `\ref{tab:rank-summary-fba}` and Table
`\ref{tab:market-rank-variation}`.

The FBA density mass is concentrated in the lower region of `rank_pct`,
with a modal value in the 0.10 to 0.15 range. The non-FBA density is
spread over the unit interval and accumulates additional mass on the
right side, where `rank_pct` approaches one and seller-list positions
are less prominent. The non-FBA group spans the full retained rank
range up to position 93, whereas the worst observed FBA position is 67.
The first two moments of these two distributions are tabulated in
Table `\ref{tab:rank-summary-fba}` (FBA mean `rank_pct` 0.194 against
non-FBA 0.574); the figure is the marginal distribution underneath
those moments and is not an estimate of an FBA ordering association.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F1. Empirical density of rank_pct by FBA status. Gaussian KDE
# evaluated on the unit interval.
# -----------------------------------------------------------------------------
from scipy.stats import gaussian_kde

x_fba = df_final.loc[df_final['fba_from_shipper'] == 1, 'rank_pct'].to_numpy()
x_non = df_final.loc[df_final['fba_from_shipper'] == 0, 'rank_pct'].to_numpy()

grid = np.linspace(0.0, 1.0, 400)
kde_fba = gaussian_kde(x_fba, bw_method='scott')(grid)
kde_non = gaussian_kde(x_non, bw_method='scott')(grid)

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.fill_between(grid, 0, kde_fba, color=COLOR_FBA_FILL,
                alpha=ALPHA_FILL, linewidth=0.0, zorder=1)
ax.fill_between(grid, 0, kde_non, color=COLOR_NONFBA_FILL,
                alpha=ALPHA_FILL, linewidth=0.0, zorder=1)
ax.plot(grid, kde_fba, color=COLOR_FBA_LINE,    linestyle='-',
        linewidth=1.6, zorder=2)
ax.plot(grid, kde_non, color=COLOR_NONFBA_LINE, linestyle='--',
        linewidth=1.6, zorder=2)
ax.set_xlim(0, 1)
ax.set_ylim(bottom=0)
ax.set_xlabel('Normalized within-market rank (rank_pct)')
ax.set_ylabel('Empirical density')

from matplotlib.legend_handler import HandlerTuple
handles, labels = _density_legend_handles(len(x_fba), len(x_non))
ax.legend(handles=handles, labels=labels,
          handler_map={tuple: HandlerTuple(ndivide=None, pad=0.6)},
          frameon=False, loc='upper right', handlelength=3.0)
ax.tick_params(direction='out', length=3)

f1_png = save_figure(fig, 'fig_F1_rank_pct_density')
print('Saved', f1_png)


---

## Figure F2. FBA share by starting-rank tier in the dynamic transition sample

LaTeX label: `fig:fba-share-by-rank-tier`. File:
`fig_F2_fba_share_by_rank_tier.pdf`. Rendered in the supplementary
chapter `\ref{app:additional-results}`, subsection
*Dynamic power and rank-tier support*, before
Table `\ref{tab:app-ch6-ranktier-stress}`.

The shares displayed in the figure are 57.4 percent in the top quartile
(1,258 observations), 9.3 percent in the middle half (2,456
observations) and 1.0 percent in the bottom quartile (1,244
observations, 12 FBA rows). The vertical reference line marks the
panel-wide FBA share of 19.5 percent. This support geometry is the
empirical basis for the support-sensitive reading of the D4
transition-by-starting-rank-tier saturation discussed in Section
`\ref{sec:robustness-ranktier}`, and for the minimum detectable
effects in Table `\ref{tab:app-ch6-dynamic-mde}`. The displayed
quantities are reproduced from Table `\ref{tab:app-ch6-ranktier-stress}`;
no estimation is performed on this object in the present cell.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F2. FBA share by starting-rank tier on the dynamic transition
# sample. Values are taken from Table \ref{tab:app-ch6-ranktier-stress}
# of the thesis (top quartile 1,258 obs, 57.4%; middle half 2,456 obs,
# 9.3%; bottom quartile 1,244 obs, 1.0%). No estimation is performed
# on this object in this cell.
# -----------------------------------------------------------------------------
tier_records = [
    {'tier': 'Top quartile',  'n_obs': 1258, 'fba_share': 0.574},
    {'tier': 'Middle half',   'n_obs': 2456, 'fba_share': 0.093},
    {'tier': 'Bottom quartile', 'n_obs': 1244, 'fba_share': 0.010},
]
tier_df = pd.DataFrame(tier_records)

panel_fba_share = 996 / 5107  # full third-party panel reference

fig, ax = plt.subplots(figsize=(5.5, 3.0))
y_positions = np.arange(len(tier_df))[::-1]
bars = ax.barh(
    y_positions, tier_df['fba_share'],
    color=COLOR_FBA_FILL, edgecolor='black',
    linewidth=0.8, height=0.55, alpha=ALPHA_FILL + 0.30,
)
for y, share, n in zip(y_positions, tier_df['fba_share'], tier_df['n_obs']):
    ax.text(share + 0.012, y, f'{share*100:.1f}% (n={n:,})',
            va='center', ha='left', fontsize=9)
ax.axvline(panel_fba_share, color='black', linestyle=':', linewidth=1.0)
ax.text(panel_fba_share + 0.005, len(tier_df) - 1.55,
        f'static-panel FBA share = {panel_fba_share*100:.1f}%',
        ha='left', va='center', fontsize=8)
ax.set_yticks(y_positions)
ax.set_yticklabels(tier_df['tier'])
ax.set_xlabel('FBA share within rank tier')
ax.set_xlim(0, 0.78)
ax.tick_params(direction='out', length=3)

f2_png = save_figure(fig, 'fig_F2_fba_share_by_rank_tier')
print('Saved', f2_png)


---

## Figure F3. Frisch-Waugh-Lovell scatter for the S4 FBA coefficient

LaTeX label: `fig:fwl-fba-s4`. File:
`fig_F3_partial_regression_plot_fba_S4.pdf`. Rendered in
appendix Chapter `\ref{app:econometric-spec}`, Section
`\ref{app:static-specs}` of `main.tex`, alongside the discussion of
the S1 to S4 nest.

The horizontal axis is FBA residualised on log seller reviews,
positive-review percentage, star rating, product price, repaired
shipping price, the contact-courier flag, robust minimum delivery
days, and market fixed effects. The vertical axis is `rank_pct`
residualised on the same set. The OLS slope through the two residual
clouds is -0.0510, that is, the S4 FBA coefficient reported in
Table `\ref{tab:ch5-static-models}`. The cluster-robust standard
error that supports the S4 hypothesis test is in the table, not in
the figure: the displayed line is a within-market control-adjusted
slope, not an inferential representation.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F3. Frisch-Waugh-Lovell scatter for the S4 FBA coefficient.
# The FWL slope through the two residual clouds equals the S4 FBA
# coefficient reported in Table \ref{tab:ch5-static-models}.
# -----------------------------------------------------------------------------
s4_controls = [
    'log1p_num_valutazioni',
    'valutazioni_positive',
    'stelle',
    'prezzo',
    'prezzo_spedizione_repaired',
    'contact_courier_flag',
    'g_cons_min_robust',
]
formula_y = 'rank_pct ~ ' + ' + '.join(s4_controls) + ' + C(market_id)'
formula_x = 'fba_from_shipper ~ ' + ' + '.join(s4_controls) + ' + C(market_id)'

res_y_aux = smf.ols(formula_y, data=df_final).fit()
res_x_aux = smf.ols(formula_x, data=df_final).fit()

e_y = np.asarray(res_y_aux.resid, dtype=float)
e_x = np.asarray(res_x_aux.resid, dtype=float)

slope_fwl = float(np.sum(e_x * e_y) / np.sum(e_x ** 2))
print(f'FWL slope (should equal S4 coef): {slope_fwl:+.6f}')

# Points coloured by the raw FBA indicator so that the negative slope
# is read against the geometry of the two groups within the residual
# cloud.
mask_fba = df_final['fba_from_shipper'].to_numpy() == 1

fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.scatter(e_x[~mask_fba], e_y[~mask_fba],
           s=5, alpha=0.30, color=COLOR_NONFBA_FILL,
           edgecolor='none', zorder=1)
ax.scatter(e_x[mask_fba], e_y[mask_fba],
           s=5, alpha=0.45, color=COLOR_FBA_FILL,
           edgecolor='none', zorder=1)
ax.axhline(0.0, color='black', linestyle=':', linewidth=0.7, zorder=2)
ax.axvline(0.0, color='black', linestyle=':', linewidth=0.7, zorder=2)

# FWL regression line through the residual cloud. Intercept is zero
# because the residuals from each auxiliary OLS have mean zero.
x_grid = np.linspace(np.min(e_x), np.max(e_x), 200)
ax.plot(x_grid, slope_fwl * x_grid, color='black', linewidth=1.6, zorder=3)

ax.set_xlabel('e(FBA): FBA residualised on S4 controls and market FE')
ax.set_ylabel('e(rank_pct): residualised on same controls')
ax.tick_params(direction='out', length=3)

# Legend keyed to the raw FBA indicator. The negative slope is supported
# by FBA points on the right side of the cloud (e(FBA) > 0) sitting at
# lower e(rank_pct), and conversely for non-FBA points on the left.
from matplotlib.lines import Line2D
n_fba_obs = int(mask_fba.sum())
n_non_obs = int((~mask_fba).sum())
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='none', markersize=7,
           markerfacecolor=COLOR_FBA_FILL, markeredgecolor='none',
           alpha=0.85,
           label=f'FBA rows (n={n_fba_obs:,})'),
    Line2D([0], [0], marker='o', linestyle='none', markersize=7,
           markerfacecolor=COLOR_NONFBA_FILL, markeredgecolor='none',
           alpha=0.85,
           label=f'non-FBA rows (n={n_non_obs:,})'),
    Line2D([0], [0], color='black', linewidth=1.6,
           label=f'OLS slope = {slope_fwl:+.4f}'),
]
ax.legend(handles=legend_handles, frameon=False, loc='lower left',
          fontsize=9, handlelength=2.0)
fig.tight_layout()

f3_png = save_figure(fig, 'fig_F3_partial_regression_plot_fba_S4')
print('Saved', f3_png)


---

## Figure F4. Classical residual diagnostics for the S4 and D1 headline models

LaTeX label: `fig:residual-diagnostics-s4-d1`. File:
`fig_F4_residual_diagnostics_S4_D1.pdf`. Rendered in the
supplementary chapter `\ref{app:additional-results}`, subsection
*Functional form and bounded outcome*, immediately after
Table `\ref{tab:app-ch6-functional-form}`.

The top row reports diagnostics for the static S4 OLS on the
$N=5{,}107$ panel with market fixed effects; the bottom row reports
the same diagnostics for the dynamic D1 OLS on the $N=4{,}958$
seller-transition panel with seller and transition fixed effects.
Reading the panels from left to right: residuals against fitted
values with a binned running mean, the normal Q-Q plot of internally
studentised residuals, and studentised residuals against leverage
with the conventional plus/minus two reference lines. The bowed
pattern at the boundary of the unit interval in the top-left panel
is the mechanical consequence of fitting a linear projection on the
bounded `rank_pct` outcome and motivates the Papke-Wooldridge
fractional-logit robustness reported in
Table `\ref{tab:app-ch6-functional-form}`. The elevated leverage in
the D1 panel reflects the seller- and transition-fixed-effect design.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F4. Classical residual diagnostics for the two headline models:
# S4 static OLS (top row, on df_final) and D1 dynamic OLS (bottom row,
# on the reconstructed transition panel). For each model: residuals
# against fitted values with a binned running mean; normal Q-Q plot of
# internally studentised residuals; studentised residuals against
# leverage with the conventional plus/minus two reference lines.
# -----------------------------------------------------------------------------
# Static S4 OLS on df_final.
formula_s4 = ('rank_pct ~ fba_from_shipper + '
              + ' + '.join(s4_controls) + ' + C(market_id)')
res_s4 = smf.ols(formula_s4, data=df_final).fit()
fitted_s4 = np.asarray(res_s4.fittedvalues, dtype=float)
resid_s4  = np.asarray(res_s4.resid, dtype=float)
influence_s4 = res_s4.get_influence()
leverage_s4 = np.asarray(influence_s4.hat_matrix_diag, dtype=float)
studentized_s4 = np.asarray(influence_s4.resid_studentized_internal, dtype=float)
print(f'S4 R^2: {res_s4.rsquared:.4f}; nobs={int(res_s4.nobs)}; '
      f'leverage max={leverage_s4.max():.4f}')

# Dynamic D1 OLS on the reconstructed transition panel. transition_id
# couples consecutive markets and enters as a fixed effect together with
# seller_id.
pivot_rank_pre = df_final.pivot_table(index='market_order', columns='seller_id',
                                      values='rank_pct')
pivot_fba_pre = df_final.pivot_table(index='market_order', columns='seller_id',
                                     values='fba_from_shipper')
_dyn_records = []
_orders = sorted(pivot_rank_pre.index)
for prev_ord, next_ord in zip(_orders[:-1], _orders[1:]):
    rank_prev = pivot_rank_pre.loc[prev_ord]
    rank_next = pivot_rank_pre.loc[next_ord]
    fba_prev = pivot_fba_pre.loc[prev_ord]
    in_prev = set(rank_prev.dropna().index)
    in_next = set(rank_next.dropna().index)
    persisting = in_prev & in_next
    n_dropouts = len(in_prev - in_next)
    transition_id = f'{int(prev_ord):02d}_to_{int(next_ord):02d}'
    for seller in persisting:
        _dyn_records.append({
            'seller_id': seller,
            'transition_id': transition_id,
            'rank_pct_improvement': float(rank_prev[seller]) - float(rank_next[seller]),
            'fba_t': int(fba_prev[seller]),
            'dropouts_total_focal': int(n_dropouts),
        })
df_d1 = pd.DataFrame(_dyn_records)
df_d1['fba_x_dropouts_total_focal'] = (
    df_d1['fba_t'].astype(float) * df_d1['dropouts_total_focal'].astype(float)
)

formula_d1 = ('rank_pct_improvement ~ fba_x_dropouts_total_focal '
              '+ C(seller_id) + C(transition_id)')
res_d1 = smf.ols(formula_d1, data=df_d1).fit()
fitted_d1 = np.asarray(res_d1.fittedvalues, dtype=float)
resid_d1  = np.asarray(res_d1.resid, dtype=float)
influence_d1 = res_d1.get_influence()
leverage_d1 = np.asarray(influence_d1.hat_matrix_diag, dtype=float)
studentized_d1 = np.asarray(influence_d1.resid_studentized_internal, dtype=float)
print(f'D1 R^2: {res_d1.rsquared:.4f}; nobs={int(res_d1.nobs)}; '
      f'leverage max={leverage_d1.max():.4f}')


def _plot_resid_vs_fitted(ax, fitted, resid, min_n=20):
    ax.scatter(fitted, resid, s=4, alpha=0.22, color='gray', edgecolor='none')
    ax.axhline(0.0, color='black', linestyle=':', linewidth=0.7)
    order = np.argsort(fitted)
    n_bins = 30
    edges = np.linspace(fitted[order][0], fitted[order][-1], n_bins + 1)
    centres = 0.5 * (edges[:-1] + edges[1:])
    mean_per_bin = []
    for k in range(n_bins):
        sel = (fitted >= edges[k]) & (fitted < edges[k + 1])
        if int(sel.sum()) >= min_n:
            mean_per_bin.append(resid[sel].mean())
        else:
            mean_per_bin.append(np.nan)
    mean_per_bin = np.asarray(mean_per_bin)
    valid = ~np.isnan(mean_per_bin)
    ax.plot(centres[valid], mean_per_bin[valid], color='black', linewidth=1.2)
    ax.set_xlabel('Fitted values', fontsize=9)
    ax.set_ylabel('Residuals', fontsize=9)
    ax.tick_params(direction='out', length=3, labelsize=8)


def _plot_qq(ax, studentized):
    osm, osr = stats.probplot(studentized, dist='norm', fit=False)
    ax.scatter(osm, osr, s=4, alpha=0.30, color='gray', edgecolor='none')
    lo, hi = osm.min(), osm.max()
    ax.plot([lo, hi], [lo, hi], color='black', linewidth=1.0)
    ax.set_xlabel('Theoretical quantiles', fontsize=9)
    ax.set_ylabel('Studentised residuals', fontsize=9)
    ax.tick_params(direction='out', length=3, labelsize=8)


def _plot_leverage(ax, leverage, studentized):
    ax.scatter(leverage, studentized, s=4, alpha=0.30, color='gray', edgecolor='none')
    ax.axhline( 2.0, color='black', linestyle='--', linewidth=0.6, alpha=0.6)
    ax.axhline(-2.0, color='black', linestyle='--', linewidth=0.6, alpha=0.6)
    ax.axhline( 0.0, color='black', linestyle=':',  linewidth=0.7)
    ax.set_xlabel('Leverage', fontsize=9)
    ax.set_ylabel('Studentised residuals', fontsize=9)
    ax.tick_params(direction='out', length=3, labelsize=8)


fig, axes = plt.subplots(2, 3, figsize=(11.0, 7.0))

# Row 1 -- S4 static OLS.
_plot_resid_vs_fitted(axes[0, 0], fitted_s4, resid_s4)
_plot_qq(axes[0, 1], studentized_s4)
_plot_leverage(axes[0, 2], leverage_s4, studentized_s4)
axes[0, 0].set_title('S4 static OLS  (n = 5,107; market FE)',
                     loc='left', fontsize=10, pad=8)

# Row 2 -- D1 dynamic OLS.
_plot_resid_vs_fitted(axes[1, 0], fitted_d1, resid_d1)
_plot_qq(axes[1, 1], studentized_d1)
_plot_leverage(axes[1, 2], leverage_d1, studentized_d1)
axes[1, 0].set_title(
    f'D1 dynamic OLS  (n = {len(df_d1):,}; seller and transition FE)',
    loc='left', fontsize=10, pad=8,
)

fig.tight_layout()

f4_png = save_figure(fig, 'fig_F4_residual_diagnostics_S4_D1')
print('Saved', f4_png)


---

## Figure F5. Mean within-market rank against product price, by FBA status

LaTeX label: `fig:price-vs-rank-by-fba`. File:
`fig_F5_price_vs_rank_by_fba.pdf`. Rendered in the supplementary
chapter `\ref{app:additional-results}`, subsection
*Logistics-value-adjusted price sensitivity*, after
Table `\ref{tab:app-logistics-value-adjusted-price-results}`.

The figure plots per-bin means of within-market `rank_pct` against
product price, with 95 percent confidence intervals and a bin-count
annotation, separately for FBA ($n=996$) and non-FBA ($n=4{,}111$)
rows. The axes are restricted to the pooled 1st-99th percentile range
of the price support. The two curves are positively sloped over the
common support and the FBA curve sits below the non-FBA curve. The
bivariate price-rank correlate visualised here is the descriptive
counterpart of the attenuation reported at the $S2 \to S3$ step in
Figure `\ref{fig:attenuation-decomposition}`. No market fixed effect
is partialled out and no other covariate is adjusted.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F5. Per-bin mean of within-market rank_pct against product
# price, by FBA status, with 95% confidence intervals on each bin.
# -----------------------------------------------------------------------------
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D

mask_fba = df_final['fba_from_shipper'] == 1

price = pd.to_numeric(df_final['prezzo'], errors='coerce')
rank = pd.to_numeric(df_final['rank_pct'], errors='coerce')
keep = price.notna() & rank.notna()
price = price[keep].to_numpy()
rank = rank[keep].to_numpy()
mask_fba_keep = mask_fba[keep].to_numpy()

# Bin edges on the pooled price support. The empirical price range
# concentrates between roughly 40 and 65 EUR, so 13 bin edges of width
# about 2 EUR are used to keep each bin populated in both groups.
bin_edges = np.linspace(40.0, 66.0, 14)
bin_centres = 0.5 * (bin_edges[:-1] + bin_edges[1:])

def _binned_mean_se(x, y, edges, min_n=10):
    rows = []
    for k in range(len(edges) - 1):
        sel = (x >= edges[k]) & (x < edges[k + 1])
        n = int(sel.sum())
        if n >= min_n:
            mu = float(np.mean(y[sel]))
            se = float(np.std(y[sel], ddof=1) / np.sqrt(n))
        else:
            mu, se = np.nan, np.nan
        rows.append({'mid': 0.5 * (edges[k] + edges[k + 1]),
                     'mean': mu, 'se': se, 'n': n})
    return pd.DataFrame(rows)

stats_fba = _binned_mean_se(price[mask_fba_keep], rank[mask_fba_keep], bin_edges)
stats_non = _binned_mean_se(price[~mask_fba_keep], rank[~mask_fba_keep], bin_edges)

print(f"F5 binned cells with n>=10 (FBA / non-FBA): "
      f"{int(stats_fba['mean'].notna().sum())} / {int(stats_non['mean'].notna().sum())}")

fig, ax = plt.subplots(figsize=(7.0, 4.0))

# Non-FBA per-bin mean curve.
mask_n = stats_non['mean'].notna()
ax.errorbar(stats_non.loc[mask_n, 'mid'], stats_non.loc[mask_n, 'mean'],
            yerr=1.96 * stats_non.loc[mask_n, 'se'],
            fmt='o', color=COLOR_NONFBA_LINE, markerfacecolor=COLOR_NONFBA_FILL,
            markersize=5.0, linewidth=0.9, capsize=2.5, elinewidth=0.9,
            zorder=2)
ax.plot(stats_non.loc[mask_n, 'mid'], stats_non.loc[mask_n, 'mean'],
        color=COLOR_NONFBA_LINE, linestyle='--', linewidth=1.4, zorder=2)

# FBA per-bin mean curve.
mask_f = stats_fba['mean'].notna()
ax.errorbar(stats_fba.loc[mask_f, 'mid'], stats_fba.loc[mask_f, 'mean'],
            yerr=1.96 * stats_fba.loc[mask_f, 'se'],
            fmt='o', color=COLOR_FBA_LINE, markerfacecolor=COLOR_FBA_FILL,
            markersize=5.0, linewidth=0.9, capsize=2.5, elinewidth=0.9,
            zorder=3)
ax.plot(stats_fba.loc[mask_f, 'mid'], stats_fba.loc[mask_f, 'mean'],
        color=COLOR_FBA_LINE, linestyle='-', linewidth=1.4, zorder=3)

# Bin-count annotations along the top: one count per group per bin.
y_ann = 1.02
for _, row_fba, row_non in zip(
        range(len(stats_fba)), stats_fba.itertuples(), stats_non.itertuples()):
    if row_fba.n + row_non.n == 0:
        continue
    ax.text(row_fba.mid, y_ann, f'{row_fba.n}', color=COLOR_FBA_LINE,
            ha='center', va='bottom', fontsize=6.5)
    ax.text(row_fba.mid, y_ann + 0.04, f'{row_non.n}', color=COLOR_NONFBA_LINE,
            ha='center', va='bottom', fontsize=6.5)

ax.set_xlim(bin_edges[0] - 0.6, bin_edges[-1] + 0.6)
ax.set_ylim(0, 1.10)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_xlabel('Product price (EUR), binned')
ax.set_ylabel('Mean within-market rank (rank_pct)  \u00b1 95% CI')

n_fba_total = int(mask_fba_keep.sum())
n_non_total = int((~mask_fba_keep).sum())
legend_handles = [
    Line2D([0], [0], color=COLOR_FBA_LINE, linestyle='-',
           marker='o', markerfacecolor=COLOR_FBA_FILL,
           markersize=6, linewidth=1.4,
           label=f'FBA (n={n_fba_total:,})'),
    Line2D([0], [0], color=COLOR_NONFBA_LINE, linestyle='--',
           marker='o', markerfacecolor=COLOR_NONFBA_FILL,
           markersize=6, linewidth=1.4,
           label=f'non-FBA (n={n_non_total:,})'),
]
ax.legend(handles=legend_handles, frameon=False, loc='lower right',
          fontsize=9, handlelength=2.5)
ax.tick_params(direction='out', length=3)
fig.tight_layout()

f5_png = save_figure(fig, 'fig_F5_price_vs_rank_by_fba')
print('Saved', f5_png)


---

## Figure F6. Propensity-score densities under the total-price overlap design

LaTeX label: `fig:propensity-density-by-fba`. File:
`fig_F6_propensity_density_by_fba.pdf`. Rendered in Chapter
`\ref{ch:robustness}`, Section `\ref{sec:robustness-common-support}`
of `main.tex`, immediately after the discussion of the residual-gap
collapse under common-support trimming.

The figure displays the kernel densities of the estimated propensity
score by FBA status, under the reference total-price linear-stars
design of Table `\ref{tab:app-ch6-propensity-overlap}`. The shaded
out-of-support tails identify rows excluded by the common-support
restriction $[0.056,\,0.975]$; the trimmed sample retains $2{,}205$
of $5{,}107$ rows. The two densities are concentrated in largely
separated regions of the unit interval, with limited mass in the
overlap area. The corresponding residual-gap collapse is in
Table `\ref{tab:app-ch6-common-support-residual-gap}` and the
per-covariate balance inside the trimmed sample is in
Table `\ref{tab:app-ch6-common-support-balance}`.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F6. Propensity-score densities by FBA status under the
# total-price linear-stars overlap design of Section
# \ref{sec:robustness-common-support}. Logit covariates: market fixed
# effects, log reviews, positive-review percentage, linear stars, total
# reconstructed price, contact-courier flag, and robust delivery days.
# -----------------------------------------------------------------------------
ps_rhs = [
    'log1p_num_valutazioni',
    'valutazioni_positive',
    'stelle',
    'prezzo_totale_reconstructed',
    'contact_courier_flag',
    'g_cons_min_robust',
    'C(market_id)',
]
ps_formula = 'fba_from_shipper ~ ' + ' + '.join(ps_rhs)
ps_data = df_final.dropna(subset=[
    'log1p_num_valutazioni', 'valutazioni_positive', 'stelle',
    'prezzo_totale_reconstructed', 'contact_courier_flag', 'g_cons_min_robust',
]).copy()

# IRLS via GLM-Binomial. Mirrors the thesis fit_logit_propensity_fast
# estimator and is numerically more stable than smf.logit Newton when
# the market fixed effects plus sparse covariates push the information
# matrix close to singular.
ps_model = smf.glm(ps_formula, data=ps_data,
                   family=sm.families.Binomial()).fit(maxiter=100, disp=0)
ps_data['propensity'] = ps_model.predict(ps_data)

p_treated = ps_data.loc[ps_data['fba_from_shipper'] == 1, 'propensity'].to_numpy()
p_control = ps_data.loc[ps_data['fba_from_shipper'] == 0, 'propensity'].to_numpy()
overlap_low  = max(p_treated.min(), p_control.min())
overlap_high = min(p_treated.max(), p_control.max())
overlap_mask = ps_data['propensity'].between(overlap_low, overlap_high)
n_overlap = int(overlap_mask.sum())
print('Propensity overlap range:', f'[{overlap_low:.3f}, {overlap_high:.3f}]')
print('Overlap rows:', n_overlap, 'of', len(ps_data))

# Persist the estimated propensity score so it can be reused for
# figures F7 and F8.
df_propensity = ps_data[['entity_time_id', 'propensity']].copy()

grid = np.linspace(0.0, 1.0, 400)
kde_t = gaussian_kde(p_treated, bw_method='scott')(grid)
kde_c = gaussian_kde(p_control, bw_method='scott')(grid)

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple

fig, ax = plt.subplots(figsize=(6.8, 3.6))

# Filled densities and coloured outlines.
ax.fill_between(grid, 0, kde_t, color=COLOR_FBA_FILL,
                alpha=ALPHA_FILL, linewidth=0.0, zorder=1)
ax.fill_between(grid, 0, kde_c, color=COLOR_NONFBA_FILL,
                alpha=ALPHA_FILL, linewidth=0.0, zorder=1)
ax.plot(grid, kde_t, color=COLOR_FBA_LINE,    linestyle='-',
        linewidth=1.6, zorder=3)
ax.plot(grid, kde_c, color=COLOR_NONFBA_LINE, linestyle='--',
        linewidth=1.6, zorder=3)

# Out-of-support tails shaded in light grey; dotted vertical guides at
# the common-support boundaries [overlap_low, overlap_high].
ax.axvspan(0.0, overlap_low,
           color=COLOR_COMMON_SUPPORT, alpha=0.35, zorder=2)
ax.axvspan(overlap_high, 1.0,
           color=COLOR_COMMON_SUPPORT, alpha=0.35, zorder=2)
ax.axvline(overlap_low,  color='black', linestyle=':', linewidth=0.9, zorder=2)
ax.axvline(overlap_high, color='black', linestyle=':', linewidth=0.9, zorder=2)

ymax = max(kde_t.max(), kde_c.max()) * 1.05
ax.set_xlim(0, 1)
ax.set_ylim(0, ymax)
ax.set_xlabel('Estimated propensity score')
ax.set_ylabel('Empirical density')
ax.tick_params(direction='out', length=3)

# Composite legend outside the axes. Each density entry combines line
# and patch; the grey patch is the out-of-support shading.
density_handles, density_labels = _density_legend_handles(
    n_fba=int((df_final['fba_from_shipper'] == 1).sum()),
    n_non=int((df_final['fba_from_shipper'] == 0).sum()),
)
legend_handles = density_handles + [
    Patch(facecolor=COLOR_COMMON_SUPPORT, alpha=0.35, edgecolor='none'),
]
legend_labels = density_labels + [
    (f'out-of-support tails\n'
     f'(common support [{overlap_low:.2f}, {overlap_high:.2f}],\n'
     f' n={n_overlap:,})'),
]
ax.legend(handles=legend_handles, labels=legend_labels,
          handler_map={tuple: HandlerTuple(ndivide=None, pad=0.6)},
          frameon=False, loc='upper left',
          bbox_to_anchor=(1.01, 1.0), borderaxespad=0,
          fontsize=8, handlelength=3.0)

f6_png = save_figure(fig, 'fig_F6_propensity_density_by_fba')
print('Saved', f6_png)

# Attach the propensity score back to df_final for use by F7 and F8.
df_final = df_final.merge(df_propensity, on='entity_time_id', how='left')
df_final['common_support_flag'] = (
    df_final['propensity'].between(overlap_low, overlap_high)
).astype('Int64')


---

## Figure F7. Non-FBA rank composition before and after the common-support trim

LaTeX label: `fig:nonfba-rank-composition-trim`. File:
`fig_F7_non_fba_rank_composition_trim.pdf`. Rendered in the
supplementary chapter `\ref{app:additional-results}`, subsection
*Overlap and balance*, after
Table `\ref{tab:app-ch6-common-support-residual-gap}`.

The two panels are the histograms of `rank_pct` among non-FBA rows
before the trim (4,111 rows) and after the trim (1,271 rows), with a
vertical guide at the sample-wide top-quartile boundary at
$\mathrm{rank\_pct}=0.25$. The non-FBA top-quartile share on the
sample-wide rank scale rises from 13.4 percent before the trim to
34.2 percent after the trim. A complementary statistic on the
conditional within-market non-FBA-only rank distribution, reported
in Section `\ref{sec:robustness-common-support}`, places 64.9 percent
of the surviving non-FBA rows in the corresponding within-market
top quartile against the 25 percent reference share implied by the
quartile construction. The two shares measure different objects and
are reported jointly in the thesis.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F7. Histograms of rank_pct among non-FBA rows, before and
# after the total-price linear-stars common-support trim. Vertical
# guide at rank_pct = 0.25 marks the sample-wide top quartile.
# -----------------------------------------------------------------------------
non_fba_full = df_final.loc[df_final['fba_from_shipper'] == 0, 'rank_pct']
non_fba_trim = df_final.loc[
    (df_final['fba_from_shipper'] == 0) & (df_final['common_support_flag'] == 1),
    'rank_pct',
]

print('Non-FBA rows, full panel:    ', len(non_fba_full))
print('Non-FBA rows, common support:', len(non_fba_trim))
top_quartile_share_full = float((non_fba_full <= 0.25).mean())
top_quartile_share_trim = float((non_fba_trim <= 0.25).mean())
print(f'Non-FBA top-quartile share (full)  : {top_quartile_share_full:.3f}')
print(f'Non-FBA top-quartile share (trimmed): {top_quartile_share_trim:.3f}')

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(6.5, 3.2), sharey=True)
bins = np.linspace(0, 1, 21)
ax_left.hist(non_fba_full, bins=bins, density=True,
             color=COLOR_FBA_FILL, edgecolor='black',
             linewidth=0.6, alpha=ALPHA_FILL + 0.30)
ax_left.set_title('Before trim', fontsize=10)
ax_left.set_xlabel('rank_pct')
ax_left.set_ylabel('Empirical density')
ax_left.set_xlim(0, 1)
ax_left.axvline(0.25, color='black', linestyle=':', linewidth=0.8)
ax_left.tick_params(direction='out', length=3)

ax_right.hist(non_fba_trim, bins=bins, density=True,
              color=COLOR_NONFBA_FILL, edgecolor='black',
              linewidth=0.6, alpha=ALPHA_FILL + 0.30)
ax_right.set_title('After common-support trim', fontsize=10)
ax_right.set_xlabel('rank_pct')
ax_right.set_xlim(0, 1)
ax_right.axvline(0.25, color='black', linestyle=':', linewidth=0.8)
ax_right.tick_params(direction='out', length=3)

# Top-right annotations with a thin bounding box, placed in axes
# coordinates so they do not collide with the bars.
ann_box = dict(boxstyle='round,pad=0.30', facecolor='white',
               edgecolor='lightgray', linewidth=0.6)
ax_left.text(0.98, 0.97,
             f'n = {len(non_fba_full):,}\n'
             f'top-quartile share = {top_quartile_share_full*100:.1f}%',
             transform=ax_left.transAxes, va='top', ha='right',
             fontsize=8, bbox=ann_box)
ax_right.text(0.98, 0.97,
              f'n = {len(non_fba_trim):,}\n'
              f'top-quartile share = {top_quartile_share_trim*100:.1f}%',
              transform=ax_right.transAxes, va='top', ha='right',
              fontsize=8, bbox=ann_box)

f7_png = save_figure(fig, 'fig_F7_non_fba_rank_composition_trim')
print('Saved', f7_png)


---

## Figure F8. Standardized mean differences before and after the common-support trim

LaTeX label: `fig:smd-before-after-trim`. File:
`fig_F8_smd_before_after_trim.pdf`. Rendered in the supplementary
chapter `\ref{app:additional-results}`, subsection
*Overlap and balance*, after Figure `\ref{fig:nonfba-rank-composition-trim}`.

The Love plot displays the per-covariate standardized mean
difference, computed as FBA minus non-FBA divided by the pooled
standard deviation, on the eight covariates of
Table `\ref{tab:balance-fba}`, before the trim ($5{,}107$ rows) and
inside the total-price linear-stars common-support subsample
($2{,}205$ rows). Vertical guides mark the conventional plus/minus
0.10 thresholds. No main commercial covariate is brought below the
conventional plus/minus 0.25 threshold by the trim, so the
trimmed sample is not a balanced overlap subset. The figure is the
balance diagnostic that motivates reading the residual-gap collapse
inside the trim as a support-re-selection effect rather than as a
treatment-effect estimate on a balanced subset.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F8. Per-covariate standardized mean differences before and
# after the total-price linear-stars common-support trim. Pooled
# standard deviation as the denominator.
# -----------------------------------------------------------------------------
balance_covariates = [
    ('log1p_num_valutazioni',       'log(1 + reviews)'),
    ('valutazioni_positive',        'positive reviews (%)'),
    ('stelle',                      'star rating'),
    ('prezzo',                      'product price (EUR)'),
    ('prezzo_spedizione_repaired',  'shipping price (EUR)'),
    ('prezzo_totale_reconstructed', 'total price (EUR)'),
    ('contact_courier_flag',        'contact-courier flag'),
    ('g_cons_min_robust',           'delivery, min days'),
]

mask_fba_all = df_final['fba_from_shipper'] == 1
mask_keep = df_final['common_support_flag'] == 1

smd_records = []
for col, label in balance_covariates:
    s_fba_full = pd.to_numeric(df_final.loc[mask_fba_all, col], errors='coerce').dropna()
    s_non_full = pd.to_numeric(df_final.loc[~mask_fba_all, col], errors='coerce').dropna()
    s_fba_trim = pd.to_numeric(df_final.loc[mask_fba_all & mask_keep, col], errors='coerce').dropna()
    s_non_trim = pd.to_numeric(df_final.loc[~mask_fba_all & mask_keep, col], errors='coerce').dropna()

    sd_full = float(np.sqrt((s_fba_full.var(ddof=1) + s_non_full.var(ddof=1)) / 2))
    sd_trim = float(np.sqrt((s_fba_trim.var(ddof=1) + s_non_trim.var(ddof=1)) / 2))

    smd_before = (s_fba_full.mean() - s_non_full.mean()) / sd_full if sd_full > 0 else 0.0
    smd_after  = (s_fba_trim.mean() - s_non_trim.mean()) / sd_trim if sd_trim > 0 else 0.0
    smd_records.append({'covariate': label, 'smd_before': smd_before, 'smd_after': smd_after})

smd_df = pd.DataFrame(smd_records)
smd_df['abs_before'] = smd_df['smd_before'].abs()
smd_df = smd_df.sort_values('abs_before').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(7.0, 4.0))
y = np.arange(len(smd_df))

# Connect the before and after points for each covariate with a thin
# grey segment.
for yi, row in zip(y, smd_df.itertuples()):
    ax.plot([row.smd_before, row.smd_after], [yi, yi],
            color='gray', linewidth=0.6, alpha=0.7, zorder=1)

ax.scatter(smd_df['smd_before'], y, s=46, marker='o',
           facecolor=COLOR_NONFBA_FILL, edgecolor=COLOR_NONFBA_LINE,
           linewidth=0.9, zorder=3, label='Before trim')
ax.scatter(smd_df['smd_after'], y, s=46, marker='s',
           facecolor=COLOR_FBA_FILL, edgecolor=COLOR_FBA_LINE,
           linewidth=0.9, zorder=3, label='After trim')

ax.axvline(0.0,  color='black', linestyle=':',  linewidth=0.7)
ax.axvline( 0.10, color='black', linestyle='--', linewidth=0.5, alpha=0.6)
ax.axvline(-0.10, color='black', linestyle='--', linewidth=0.5, alpha=0.6)

ax.set_yticks(y)
ax.set_yticklabels(smd_df['covariate'])
ax.set_xlabel('Standardized mean difference (FBA \u2212 non-FBA)')
ax.legend(frameon=False, loc='upper right', fontsize=9)
ax.tick_params(direction='out', length=3)

f8_png = save_figure(fig, 'fig_F8_smd_before_after_trim')
print('Saved', f8_png)


---

## Figure F9. Distribution of dropouts_total_focal by focal FBA status

LaTeX label: `fig:dropouts-distribution-by-fba`. File:
`fig_F9_dropouts_distribution_by_focal_fba.pdf`. Rendered in the
supplementary chapter `\ref{app:additional-results}`, subsection
*Dynamic sample support and turnover exposure*.

The figure displays the empirical distribution of
`dropouts_total_focal` separately for transitions in which the
focal seller is FBA ($n=962$) or non-FBA ($n=3{,}996$) at the lagged
market, on the continuing seller-transition panel ($N=4{,}958$);
vertical guides mark the per-group medians. The two distributions
are visually indistinguishable. The D1 interaction coefficient on
$\mathrm{FBA}\times\mathrm{dropouts}$ in
Table `\ref{tab:ch5-dynamic-models}` therefore identifies a
per-unit-exposure differential between the two fulfillment groups,
not a level shift in the exposure variable itself.


In [ ]:
# -----------------------------------------------------------------------------
# Dynamic transition panel, used by F9 and F10.
# A seller-transition row exists when a seller is present in two
# consecutive retained markets. dropouts_total_focal counts all sellers
# that were in the lagged market and disappear from the lead market.
# -----------------------------------------------------------------------------
pivot_rank = df_final.pivot_table(index='market_order', columns='seller_id',
                                  values='rank_pct')
pivot_fba = df_final.pivot_table(index='market_order', columns='seller_id',
                                 values='fba_from_shipper')

dyn_records = []
orders = sorted(pivot_rank.index)
for prev_ord, next_ord in zip(orders[:-1], orders[1:]):
    rank_prev = pivot_rank.loc[prev_ord]
    rank_next = pivot_rank.loc[next_ord]
    fba_prev = pivot_fba.loc[prev_ord]

    in_prev = set(rank_prev.dropna().index)
    in_next = set(rank_next.dropna().index)
    dropouts = in_prev - in_next
    persisting = in_prev & in_next
    n_dropouts = len(dropouts)

    for seller in persisting:
        dyn_records.append({
            'seller_id': seller,
            'm_t': int(prev_ord),
            'm_tp1': int(next_ord),
            'rank_pct_t': float(rank_prev[seller]),
            'rank_pct_tp1': float(rank_next[seller]),
            'rank_pct_improvement': float(rank_prev[seller]) - float(rank_next[seller]),
            'fba_t': int(fba_prev[seller]),
            'dropouts_total_focal': int(n_dropouts),
        })

df_dynamic = pd.DataFrame(dyn_records)
df_dynamic['fba_x_dropouts_total_focal'] = (
    df_dynamic['fba_t'] * df_dynamic['dropouts_total_focal']
)
print('Dynamic transition panel:')
print(f'  rows = {len(df_dynamic):,}')
print(f'  transitions = {df_dynamic[["m_t", "m_tp1"]].drop_duplicates().shape[0]}')
print(f'  focal FBA rows     = {int(df_dynamic["fba_t"].sum()):,}')
print(f'  focal non-FBA rows = {int((1 - df_dynamic["fba_t"]).sum()):,}')

# -----------------------------------------------------------------------------
# Figure F9. Side-by-side histograms of dropouts_total_focal for
# transitions with focal FBA and focal non-FBA sellers at the lagged
# market. Vertical guides mark the per-group medians.
# -----------------------------------------------------------------------------
d_fba = df_dynamic.loc[df_dynamic['fba_t'] == 1, 'dropouts_total_focal'].to_numpy()
d_non = df_dynamic.loc[df_dynamic['fba_t'] == 0, 'dropouts_total_focal'].to_numpy()

dmin, dmax = int(min(d_fba.min(), d_non.min())), int(max(d_fba.max(), d_non.max()))
bins = np.arange(dmin, dmax + 2) - 0.5

fig, ax = plt.subplots(figsize=(6.5, 3.6))
hist_f, _ = np.histogram(d_fba, bins=bins, density=True)
hist_n, _ = np.histogram(d_non, bins=bins, density=True)
centres = 0.5 * (bins[:-1] + bins[1:])
bar_width = 0.42
ax.bar(centres - bar_width / 2, hist_f, width=bar_width,
       color=COLOR_FBA_FILL, edgecolor=COLOR_FBA_LINE, linewidth=0.6,
       alpha=ALPHA_FILL + 0.30, zorder=2, label='focal FBA')
ax.bar(centres + bar_width / 2, hist_n, width=bar_width,
       color=COLOR_NONFBA_FILL, edgecolor=COLOR_NONFBA_LINE, linewidth=0.6,
       alpha=ALPHA_FILL + 0.30, zorder=2, label='focal non-FBA')

med_f, med_n = float(np.median(d_fba)), float(np.median(d_non))
ax.axvline(med_f, color=COLOR_FBA_LINE,    linestyle='-',  linewidth=1.0, alpha=0.8)
ax.axvline(med_n, color=COLOR_NONFBA_LINE, linestyle='--', linewidth=1.0, alpha=0.8)
ax.set_xlim(dmin - 0.6, dmax + 0.6)
ax.set_ylim(bottom=0)
ax.set_xlabel('dropouts_total_focal (sellers disappearing per transition)')
ax.set_ylabel('Empirical density')

handles, labels = _density_legend_handles(
    n_fba=len(d_fba), n_non=len(d_non), include_median=True,
)
# Override the default legend labels with the focal-FBA phrasing used
# in the thesis.
labels = [f'focal FBA (n={len(d_fba):,})',
          f'focal non-FBA (n={len(d_non):,})',
          'per-group median']
ax.legend(handles=handles, labels=labels,
          handler_map={tuple: HandlerTuple(ndivide=None, pad=0.6)},
          frameon=False, loc='upper right', fontsize=9, handlelength=3.0)
ax.tick_params(direction='out', length=3)

f9_png = save_figure(fig, 'fig_F9_dropouts_distribution_by_focal_fba')
print('Saved', f9_png)


---

## Figure F10. Rank improvement and exposure to dropouts, by focal FBA status

LaTeX label: `fig:dynamic-means-by-fba`. File:
`fig_F10_dynamic_means_by_focal_fba.pdf`. Rendered in Chapter
`\ref{ch:results}`, Section `\ref{sec:results-dynamic}` of
`main.tex`, immediately after Table `\ref{tab:ch5-dynamic-models}`.

The top panel displays per-level binned means of
`rank_pct_improvement` against `dropouts_total_focal`, separately
for transitions with focal FBA ($n=962$) and focal non-FBA
($n=3{,}996$) sellers at the lagged market, with 95 percent
confidence intervals using ordinary group standard errors. The
bottom panel displays the per-level FBA minus non-FBA difference
with the pooled-variance confidence interval. The configuration is
the descriptive analogue of the D1 interaction coefficient of
$0.002467$ ($p=0.0020$) reported in
Table `\ref{tab:ch5-dynamic-models}`. The clustered-SE inference
that supports the D1 hypothesis test is in the table; the
per-level intervals in the figure are reported for visual
orientation only.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F10. Per-level binned means of rank_pct_improvement against
# dropouts_total_focal, separately for focal-FBA and focal-non-FBA
# rows, with 95% group confidence intervals (top panel). The bottom
# panel reports the per-level FBA minus non-FBA difference with the
# pooled-variance confidence interval. The slope difference between
# the two top-panel curves is the descriptive analogue of the D1
# interaction coefficient reported in
# Table \ref{tab:ch5-dynamic-models}.
# -----------------------------------------------------------------------------
y_imp = df_dynamic['rank_pct_improvement'].to_numpy()
x_drop = df_dynamic['dropouts_total_focal'].to_numpy()
fba_t = df_dynamic['fba_t'].to_numpy().astype(bool)

levels = sorted(np.unique(x_drop).astype(int).tolist())
group_stats = []
for k in levels:
    fba_at_k = y_imp[(x_drop == k) & fba_t]
    non_at_k = y_imp[(x_drop == k) & ~fba_t]
    n_f, n_n = len(fba_at_k), len(non_at_k)
    mu_f = float(np.mean(fba_at_k)) if n_f else np.nan
    mu_n = float(np.mean(non_at_k)) if n_n else np.nan
    se_f = float(np.std(fba_at_k, ddof=1) / np.sqrt(n_f)) if n_f > 1 else np.nan
    se_n = float(np.std(non_at_k, ddof=1) / np.sqrt(n_n)) if n_n > 1 else np.nan
    diff = mu_f - mu_n if (n_f and n_n) else np.nan
    se_diff = np.sqrt(se_f ** 2 + se_n ** 2) if (n_f > 1 and n_n > 1) else np.nan
    group_stats.append({'k': k, 'n_fba': n_f, 'n_non': n_n,
                        'mean_fba': mu_f, 'mean_non': mu_n,
                        'se_fba': se_f, 'se_non': se_n,
                        'diff': diff, 'se_diff': se_diff})
stats_dyn = pd.DataFrame(group_stats)
print('F10 group cells (k, n_fba, n_non):')
print(stats_dyn[['k', 'n_fba', 'n_non', 'mean_fba', 'mean_non', 'diff']]
      .round(5).to_string(index=False))

fig, (ax_main, ax_diff) = plt.subplots(
    2, 1, figsize=(7.0, 6.0), sharex=True,
    gridspec_kw={'height_ratios': [3.2, 1.4], 'hspace': 0.10},
)

# Top panel. Per-group means and 95% confidence intervals per integer
# level of dropouts_total_focal.
offset = 0.10
ax_main.errorbar(stats_dyn['k'] - offset, stats_dyn['mean_non'],
                 yerr=1.96 * stats_dyn['se_non'],
                 fmt='o', color=COLOR_NONFBA_LINE,
                 markerfacecolor=COLOR_NONFBA_FILL,
                 markersize=6.0, linewidth=0, capsize=3, elinewidth=1.0,
                 zorder=3)
ax_main.errorbar(stats_dyn['k'] + offset, stats_dyn['mean_fba'],
                 yerr=1.96 * stats_dyn['se_fba'],
                 fmt='o', color=COLOR_FBA_LINE,
                 markerfacecolor=COLOR_FBA_FILL,
                 markersize=6.0, linewidth=0, capsize=3, elinewidth=1.0,
                 zorder=3)
# Connecting lines on the original (un-shifted) integer grid.
ax_main.plot(stats_dyn['k'], stats_dyn['mean_non'],
             color=COLOR_NONFBA_LINE, linestyle='--', linewidth=1.2, zorder=2)
ax_main.plot(stats_dyn['k'], stats_dyn['mean_fba'],
             color=COLOR_FBA_LINE, linestyle='-',  linewidth=1.2, zorder=2)

ax_main.axhline(0.0, color='black', linestyle=':', linewidth=0.7, zorder=1)
ax_main.set_ylabel('Mean rank_pct improvement  \u00b1 95% CI', fontsize=9)
ax_main.tick_params(direction='out', length=3)

# Padded top so the per-cell sample-size annotations do not collide
# with the error-bar caps.
y_data_max = float((stats_dyn['mean_fba'] + 1.96 * stats_dyn['se_fba']).max())
y_data_min = float((stats_dyn['mean_non'] - 1.96 * stats_dyn['se_non']).min())
y_pad = 0.0040
ax_main.set_ylim(y_data_min - y_pad, y_data_max + 2.5 * y_pad)
y_ann = y_data_max + 1.5 * y_pad

for _, row in stats_dyn.iterrows():
    ax_main.text(row['k'] - offset, y_ann, f'{int(row["n_non"])}',
                 color=COLOR_NONFBA_LINE, ha='center', va='center', fontsize=7)
    ax_main.text(row['k'] + offset, y_ann, f'{int(row["n_fba"])}',
                 color=COLOR_FBA_LINE,    ha='center', va='center', fontsize=7)

n_fba_total = int(fba_t.sum())
n_non_total = int((~fba_t).sum())
legend_handles = [
    Line2D([0], [0], color=COLOR_FBA_LINE, linestyle='-',
           marker='o', markerfacecolor=COLOR_FBA_FILL,
           markersize=6, linewidth=1.2,
           label=f'focal FBA (n={n_fba_total:,})'),
    Line2D([0], [0], color=COLOR_NONFBA_LINE, linestyle='--',
           marker='o', markerfacecolor=COLOR_NONFBA_FILL,
           markersize=6, linewidth=1.2,
           label=f'focal non-FBA (n={n_non_total:,})'),
]
# Bottom-left legend placement: away from the top sample-size labels
# and from the wider error bars in the upper-exposure cells.
ax_main.legend(handles=legend_handles, frameon=False, loc='lower left',
               fontsize=9, handlelength=2.5)

# Bottom panel. Per-level FBA minus non-FBA difference with 95%
# pooled-variance confidence interval.
ax_diff.errorbar(stats_dyn['k'], stats_dyn['diff'],
                 yerr=1.96 * stats_dyn['se_diff'],
                 fmt='o', color='black', markerfacecolor='black',
                 markersize=5.0, linewidth=0, capsize=3, elinewidth=1.0,
                 zorder=3)
ax_diff.plot(stats_dyn['k'], stats_dyn['diff'],
             color='black', linewidth=1.0, zorder=2)
ax_diff.axhline(0.0, color='black', linestyle=':', linewidth=0.7, zorder=1)
ax_diff.set_xlabel('dropouts_total_focal (sellers disappearing per transition)')
ax_diff.set_ylabel('FBA \u2212 non-FBA  \u00b1 95% CI', fontsize=9)
ax_diff.tick_params(direction='out', length=3)

ax_main.set_xticks(levels)
ax_diff.set_xticks(levels)
ax_main.set_xlim(min(levels) - 0.5, max(levels) + 0.5)
fig.tight_layout()

f10_png = save_figure(fig, 'fig_F10_dynamic_means_by_focal_fba')
print('Saved', f10_png)


---

## Figure F11. Covariate distributional profile by FBA status

LaTeX label: `fig:covariate-profile-by-fba`. File:
`fig_F11_covariate_profile_by_fba.pdf`. Rendered in appendix
Chapter `\ref{app:audit}`, Section `\ref{app:final-panel-structure}`
of `main.tex`, between Table `\ref{tab:appendix-topk-support}` and
the heatmaps of Figure `\ref{fig:rank-heatmap-price-stars}`.

The figure displays the empirical densities of the six covariates
entering the S4 specification of Table `\ref{tab:ch5-static-models}`,
separately for FBA ($n=996$) and non-FBA ($n=4{,}111$) rows, with
per-group median guides. Product-price panels are restricted to the
pooled 1st-99th percentile range to keep the visual scale readable.
The displayed shifts are descriptive of the sample composition by
fulfillment status. The corresponding standardized mean differences
are reported in Table `\ref{tab:balance-fba}`; the figure does not
test exchangeability and is not the balance diagnostic on the trimmed
sample.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F11. Per-covariate empirical densities by FBA status, in a
# 2x3 grid of small multiples covering the six S4 covariates. Discrete
# variables (star rating) use side-by-side histograms; continuous
# variables use Gaussian KDE. Per-group medians are marked with
# vertical guides.
# -----------------------------------------------------------------------------
from scipy.stats import gaussian_kde

covariates_for_panel = [
    {'col': 'prezzo',                       'xlabel': 'Product price (EUR)\nexcl. shipping',
     'kind': 'kde'},
    {'col': 'prezzo_totale_reconstructed',  'xlabel': 'Total price (EUR)\nproduct + shipping',
     'kind': 'kde'},
    {'col': 'log1p_num_valutazioni',        'xlabel': 'log(1 + reviews)',
     'kind': 'kde'},
    {'col': 'valutazioni_positive',         'xlabel': 'Positive reviews (%)',
     'kind': 'kde'},
    {'col': 'stelle',                       'xlabel': 'Star rating',
     'kind': 'discrete'},
    {'col': 'g_cons_min_robust',            'xlabel': 'Delivery, min days',
     'kind': 'kde'},
]

fig, axes = plt.subplots(2, 3, figsize=(9.0, 5.2))
axes = axes.ravel()

for ax, meta in zip(axes, covariates_for_panel):
    col = meta['col']
    series_fba = pd.to_numeric(df_final.loc[df_final['fba_from_shipper'] == 1, col],
                               errors='coerce').dropna().to_numpy()
    series_non = pd.to_numeric(df_final.loc[df_final['fba_from_shipper'] == 0, col],
                               errors='coerce').dropna().to_numpy()

    pooled = np.concatenate([series_fba, series_non])
    x_lo, x_hi = np.percentile(pooled, [1, 99])
    if x_hi <= x_lo:
        x_lo, x_hi = float(pooled.min()), float(pooled.max())

    if meta['kind'] == 'discrete' or np.std(series_fba) == 0 or np.std(series_non) == 0:
        # Discrete-like variable (for example star rating) or zero
        # variance in one group (for example FBA shipping always free).
        # Use a side-by-side histogram on the empirical support.
        support_values = np.unique(pooled)
        if len(support_values) <= 12:
            bins = np.r_[support_values - 0.05, support_values[-1] + 0.05]
            width = (support_values[1] - support_values[0]) / 2.5 if len(support_values) > 1 else 0.2
            hist_f, _ = np.histogram(series_fba, bins=bins, density=True)
            hist_n, _ = np.histogram(series_non, bins=bins, density=True)
            ax.bar(support_values - width / 2, hist_f, width=width,
                   color=COLOR_FBA_FILL, edgecolor='black',
                   linewidth=0.4, alpha=ALPHA_FILL + 0.30, label='FBA')
            ax.bar(support_values + width / 2, hist_n, width=width,
                   color=COLOR_NONFBA_FILL, edgecolor='black',
                   linewidth=0.4, alpha=ALPHA_FILL + 0.30, label='non-FBA')
            ax.set_xlim(support_values.min() - 0.4, support_values.max() + 0.4)
        else:
            bins = np.linspace(x_lo, x_hi, 20)
            ax.hist(series_fba, bins=bins, density=True, color=COLOR_FBA_FILL,
                    edgecolor='black', linewidth=0.4, alpha=ALPHA_FILL + 0.30)
            ax.hist(series_non, bins=bins, density=True, color=COLOR_NONFBA_FILL,
                    edgecolor='black', linewidth=0.4, alpha=ALPHA_FILL + 0.30)
            ax.set_xlim(x_lo, x_hi)
    else:
        # Continuous variable: Gaussian KDE on the 1st-99th percentile
        # range of the pooled support.
        grid = np.linspace(x_lo, x_hi, 300)
        kde_f = gaussian_kde(series_fba, bw_method='scott')(grid)
        kde_n = gaussian_kde(series_non, bw_method='scott')(grid)
        ax.fill_between(grid, 0, kde_f, color=COLOR_FBA_FILL,
                        alpha=ALPHA_FILL, linewidth=0.0, zorder=1)
        ax.fill_between(grid, 0, kde_n, color=COLOR_NONFBA_FILL,
                        alpha=ALPHA_FILL, linewidth=0.0, zorder=1)
        ax.plot(grid, kde_f, color=COLOR_FBA_LINE,    linestyle='-',
                linewidth=1.2, zorder=2)
        ax.plot(grid, kde_n, color=COLOR_NONFBA_LINE, linestyle='--',
                linewidth=1.2, zorder=2)
        ax.set_xlim(x_lo, x_hi)

    med_f = float(np.median(series_fba))
    med_n = float(np.median(series_non))
    ax.axvline(med_f, color=COLOR_FBA_LINE,    linestyle='-',
               linewidth=0.9, alpha=0.7)
    ax.axvline(med_n, color=COLOR_NONFBA_LINE, linestyle='--',
               linewidth=0.9, alpha=0.7)

    ax.set_xlabel(meta['xlabel'], fontsize=9)
    ax.set_ylim(bottom=0)
    ax.set_yticks([])
    ax.tick_params(direction='out', length=3, labelsize=8)

axes[0].set_ylabel('Empirical density', fontsize=9)
axes[3].set_ylabel('Empirical density', fontsize=9)

from matplotlib.legend_handler import HandlerTuple
handles, labels = _density_legend_handles(
    n_fba=int((df_final['fba_from_shipper'] == 1).sum()),
    n_non=int((df_final['fba_from_shipper'] == 0).sum()),
    include_median=True,
)
fig.legend(handles=handles, labels=labels,
           handler_map={tuple: HandlerTuple(ndivide=None, pad=0.6)},
           frameon=False, ncol=3,
           loc='lower center', bbox_to_anchor=(0.5, -0.02),
           fontsize=9, handlelength=3.0)
fig.tight_layout(rect=(0, 0.04, 1, 1))

f11_png = save_figure(fig, 'fig_F11_covariate_profile_by_fba')
print('Saved', f11_png)


---

## Figure F12. Panel structure across the 62 markets

LaTeX label: `fig:panel-structure-timeline`. File:
`fig_F12_panel_structure_timeline.pdf`. Rendered in Chapter
`\ref{ch:data}`, Section `\ref{sec:data-descriptive}` of `main.tex`,
immediately after Table `\ref{tab:market-rank-variation}`.

The stacked bars on the left axis report retained FBA and non-FBA
seller-market observations within each chronologically ordered
market; the connected dots on the right axis report the within-market
FBA share. The horizontal reference marks the panel-wide FBA share
of 19.5 percent. Retained market sizes range from 72 to 93 third-party
sellers (Table `\ref{tab:market-rank-variation}`), and the FBA share
is consistently a minority within each snapshot. The panel composition
is the descriptive precondition for the within-market design adopted
in Chapter `\ref{ch:methodology}`; higher-moment stability of the
composition is documented in
Figure `\ref{fig:market-structure-stability}`.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F12. Stacked seller counts (left axis) and within-market FBA
# share (right axis) across the 62 chronologically ordered markets.
# Horizontal reference at the panel-wide FBA share.
# -----------------------------------------------------------------------------
market_struct = (
    df_final
    .groupby(['market_order', 'market_id'], as_index=False)
    .agg(n_sellers=('seller_id', 'size'),
         n_fba=('fba_from_shipper', 'sum'))
    .sort_values('market_order')
    .reset_index(drop=True)
)
market_struct['n_non_fba'] = market_struct['n_sellers'] - market_struct['n_fba']
market_struct['fba_share'] = market_struct['n_fba'] / market_struct['n_sellers']

fig, ax_bar = plt.subplots(figsize=(8.0, 3.6))

x = market_struct['market_order'].to_numpy()
bar_width = 0.85
ax_bar.bar(x, market_struct['n_non_fba'],
           color=COLOR_NONFBA_FILL, edgecolor='black', linewidth=0.3,
           width=bar_width, label='non-FBA sellers', alpha=ALPHA_FILL + 0.30)
ax_bar.bar(x, market_struct['n_fba'],
           bottom=market_struct['n_non_fba'],
           color=COLOR_FBA_FILL, edgecolor='black', linewidth=0.3,
           width=bar_width, label='FBA sellers', alpha=ALPHA_FILL + 0.30)

ax_bar.set_xlabel('Market index (chronological order, 62 markets)')
ax_bar.set_ylabel('Number of sellers per market')
ax_bar.set_xlim(0.5, len(market_struct) + 0.5)
ax_bar.set_ylim(0, market_struct['n_sellers'].max() * 1.15)
ax_bar.tick_params(direction='out', length=3)

# Right axis. Within-market FBA share, with a dotted reference at the
# panel-wide share.
ax_share = ax_bar.twinx()
ax_share.plot(x, market_struct['fba_share'],
              color=COLOR_FBA_LINE, linewidth=1.4, marker='o', markersize=3.2,
              markeredgecolor=COLOR_FBA_LINE, markerfacecolor=COLOR_FBA_LINE,
              label='FBA share (right axis)')
panel_share = market_struct['n_fba'].sum() / market_struct['n_sellers'].sum()
ax_share.axhline(panel_share, color='black', linestyle=':', linewidth=0.9,
                 label=f'overall FBA share = {panel_share*100:.1f}%')
ax_share.set_ylabel('FBA share within market')
ax_share.set_ylim(0, max(0.55, market_struct['fba_share'].max() * 1.20))
ax_share.tick_params(direction='out', length=3)

# Hide the top spine of both axes for a cleaner appearance.
ax_bar.spines['top'].set_visible(False)
ax_share.spines['top'].set_visible(False)

# Combined legend placed below the plot to keep the upper-left corner
# clear of the FBA-share line.
handles_bar, labels_bar = ax_bar.get_legend_handles_labels()
handles_share, labels_share = ax_share.get_legend_handles_labels()
ax_bar.legend(handles_bar + handles_share, labels_bar + labels_share,
              frameon=False, loc='upper center',
              bbox_to_anchor=(0.5, -0.18), ncol=4, fontsize=8)

f12_png = save_figure(fig, 'fig_F12_panel_structure_timeline')
print('Saved', f12_png)


---

## Figure F13. Mean within-market rank in the price plane, by FBA status

LaTeX labels: `fig:rank-heatmap-price-stars`,
`fig:rank-heatmap-price-delivery`, `fig:rank-heatmap-price-shipping`.
Files: `fig_F13a_heatmap_price_stars.pdf`,
`fig_F13b_heatmap_price_delivery.pdf`,
`fig_F13c_heatmap_price_shipping.pdf`. Rendered as three adjacent
floats in appendix Chapter `\ref{app:audit}`, Section
`\ref{app:final-panel-structure}` of `main.tex`, after
Figure `\ref{fig:covariate-profile-by-fba}`.

Each heatmap reports the mean within-market `rank_pct` in bivariate
cells defined by product-price quartile (x-axis) and a second
covariate band (y-axis), separately for FBA (left) and non-FBA
(right). Darker cells correspond to better mean positions, that is,
lower `rank_pct`; cell counts are annotated within each cell. The
FBA sub-grid is empty below the four-star band in panel (a) by
construction of the panel sample; the within-cell FBA advantage in
panel (b) is not mechanically eliminated by stratification on
delivery days; in panel (c), FBA rows fall only in the free-shipping
band, in line with the standardized mean difference of $-0.963$ on
repaired shipping price reported in Table `\ref{tab:balance-fba}`.
The three heatmaps display bivariate conditional means without market
fixed effects.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F13. Three joint heatmaps of mean within-market rank_pct by
# FBA status, rendered as three independent figure files. Each heatmap
# crosses product-price quartile (x-axis) with a second covariate
# banded into three groups (y-axis): star rating (F13a),
# delivery days (F13b), shipping price (F13c). A shared helper avoids
# code duplication across the three panels.
# -----------------------------------------------------------------------------
price_q_raw = pd.to_numeric(df_final['prezzo'], errors='coerce')
rank_h_raw  = pd.to_numeric(df_final['rank_pct'], errors='coerce')
fba_h_raw   = df_final['fba_from_shipper']

# Common price-quartile edges on the pooled support.
price_edges = np.quantile(price_q_raw.dropna(), [0.0, 0.25, 0.5, 0.75, 1.0])
price_labels = [f'Q{i+1}\n[{price_edges[i]:.0f},{price_edges[i+1]:.0f}]'
                for i in range(4)]


def _build_heatmap_df(second_col, second_edges, second_labels):
    # Return (grid_fba, grid_non), each a wide-format grid with rows
    # indexed by second_q and columns indexed by price_q.
    col2 = pd.to_numeric(df_final[second_col], errors='coerce')
    keep = price_q_raw.notna() & col2.notna() & rank_h_raw.notna()
    base = pd.DataFrame({
        'prezzo':   price_q_raw[keep].to_numpy(),
        'second':   col2[keep].to_numpy(),
        'rank_pct': rank_h_raw[keep].to_numpy(),
        'fba':      fba_h_raw[keep].to_numpy(),
    })
    base['price_q']  = pd.cut(base['prezzo'],  bins=price_edges,
                               labels=price_labels, include_lowest=True)
    base['second_q'] = pd.cut(base['second'], bins=second_edges,
                               labels=second_labels, include_lowest=True)

    def _grid(d):
        return d.groupby(['second_q', 'price_q'], observed=False).agg(
            mean_rank=('rank_pct', 'mean'), n=('rank_pct', 'size'),
        ).unstack()

    return _grid(base[base['fba'] == 1]), _grid(base[base['fba'] == 0])


def _draw_heatmap_pair(axes, grid_fba, grid_non, y_labels, y_title,
                       cmap='Oranges_r'):
    all_vals = np.concatenate([
        grid_fba['mean_rank'].to_numpy().ravel(),
        grid_non['mean_rank'].to_numpy().ravel(),
    ])
    vmin = float(np.nanmin(all_vals))
    vmax = float(np.nanmax(all_vals))
    im = None
    for ax, grid, title in [(axes[0], grid_fba, 'FBA'),
                             (axes[1], grid_non, 'non-FBA')]:
        mean_mat = grid['mean_rank'].to_numpy().astype(float)
        n_mat    = grid['n'].to_numpy().astype(float)
        im = ax.imshow(mean_mat, cmap=cmap, vmin=vmin, vmax=vmax,
                       aspect='auto')
        for i in range(mean_mat.shape[0]):
            for j in range(mean_mat.shape[1]):
                val = mean_mat[i, j]
                n_c = n_mat[i, j]
                if np.isfinite(val) and n_c > 0:
                    colour = ('white'
                              if (vmax - val) / (vmax - vmin + 1e-9) > 0.55
                              else 'black')
                    ax.text(j, i, f'{val:.2f}\n(n={int(n_c)})',
                            ha='center', va='center',
                            fontsize=7.5, color=colour)
        ax.set_xticks(np.arange(mean_mat.shape[1]))
        ax.set_xticklabels(price_labels, fontsize=8)
        ax.set_yticks(np.arange(len(y_labels)))
        ax.set_yticklabels(y_labels, fontsize=9)
        ax.set_xlabel('Product price quartile (EUR, excl. shipping)', fontsize=9)
        ax.set_title(title, loc='left', fontsize=10)
    axes[0].set_ylabel(y_title, fontsize=9)
    return im


# Heatmap (a). Price quartile by star-rating band.
star_edges  = [2.5, 3.99, 4.49, 5.01]
star_labels = ['<4', '4-4.5', '5']
g_fba_s, g_non_s = _build_heatmap_df('stelle', star_edges, star_labels)

fig_a, axes_a = plt.subplots(1, 2, figsize=(10.0, 3.5))
im_a = _draw_heatmap_pair(axes_a, g_fba_s, g_non_s,
                          star_labels, 'Star rating band')
cbar_a = fig_a.colorbar(im_a, ax=axes_a, shrink=0.7, pad=0.02, location='right')
cbar_a.set_label('Mean rank_pct (darker = better)', fontsize=8)
cbar_a.ax.tick_params(labelsize=7)

f13a_png = save_figure(fig_a, 'fig_F13a_heatmap_price_stars')
print('Saved', f13a_png)

# Heatmap (b). Price quartile by minimum-delivery-day band.
deliv_pool = pd.to_numeric(df_final['g_cons_min_robust'], errors='coerce').dropna()
d_lo, d_mid = float(np.percentile(deliv_pool, 33)), float(np.percentile(deliv_pool, 67))
deliv_edges  = [0, d_lo, d_mid, float(deliv_pool.max()) + 1]
deliv_labels = [f'fast\n(\u22642 d)', f'mid\n({int(d_lo)+1}-{int(d_mid)} d)',
                f'slow\n(>{int(d_mid)} d)']
# Replace the placeholder text labels with bin labels expressed in
# days, derived from the 33rd and 67th percentiles of the pooled
# delivery support.
d_p33 = int(np.percentile(deliv_pool, 33))
d_p67 = int(np.percentile(deliv_pool, 67))
deliv_edges  = [0, d_p33 + 0.5, d_p67 + 0.5, float(deliv_pool.max()) + 1]
deliv_labels = [f'\u2264{d_p33} days', f'{d_p33+1}-{d_p67} days',
                f'>{d_p67} days']
g_fba_d, g_non_d = _build_heatmap_df('g_cons_min_robust', deliv_edges, deliv_labels)

fig_b, axes_b = plt.subplots(1, 2, figsize=(10.0, 3.5))
im_b = _draw_heatmap_pair(axes_b, g_fba_d, g_non_d,
                          deliv_labels, 'Min delivery band')
cbar_b = fig_b.colorbar(im_b, ax=axes_b, shrink=0.7, pad=0.02, location='right')
cbar_b.set_label('Mean rank_pct (darker = better)', fontsize=8)
cbar_b.ax.tick_params(labelsize=7)

f13b_png = save_figure(fig_b, 'fig_F13b_heatmap_price_delivery')
print('Saved', f13b_png)

# Heatmap (c). Price quartile by repaired-shipping-price band.
# FBA shipping is always 0 EUR in the panel by construction of the
# displayed shipping component, so the bands are chosen to span the
# non-FBA support (0 EUR to approximately 21 EUR).
ship_pool = pd.to_numeric(df_final['prezzo_spedizione_repaired'],
                           errors='coerce').dropna()
ship_edges  = [-0.01, 0.01, 5.0, float(ship_pool.max()) + 1]
ship_labels = ['free\n(0 EUR)', 'low\n(0-5 EUR)', 'high\n(>5 EUR)']
g_fba_p, g_non_p = _build_heatmap_df('prezzo_spedizione_repaired',
                                       ship_edges, ship_labels)

fig_c, axes_c = plt.subplots(1, 2, figsize=(10.0, 3.5))
im_c = _draw_heatmap_pair(axes_c, g_fba_p, g_non_p,
                          ship_labels, 'Shipping price band')
cbar_c = fig_c.colorbar(im_c, ax=axes_c, shrink=0.7, pad=0.02, location='right')
cbar_c.set_label('Mean rank_pct (darker = better)', fontsize=8)
cbar_c.ax.tick_params(labelsize=7)

f13c_png = save_figure(fig_c, 'fig_F13c_heatmap_price_shipping')
print('Saved', f13c_png)


---

## Figure F14. Within-market structural stability across the 62 snapshots

LaTeX label: `fig:market-structure-stability`. File:
`fig_F14_market_structure_stability.pdf`. Rendered in appendix
Chapter `\ref{app:audit}`, Section `\ref{app:final-panel-structure}`
of `main.tex`, after the heatmaps of Figure F13.

The left panel reports the within-market interquartile range of
product price across the 62 chronologically ordered markets; the
right panel reports the within-market mean robust minimum delivery
time in days, separately for FBA and non-FBA rows. The dotted
reference line in the left panel marks the panel-wide mean IQR
(approximately 11.5 EUR). The price-IQR series shifts to a
higher-IQR segment in the second half of the observation window;
the delivery profile is roughly stable across markets, with FBA
consistently faster than non-FBA. The two moments are descriptive
of the panel and are not tests of strict exchangeability across
snapshots.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F14. Within-market structural stability across the 62
# snapshots. Left panel: within-market interquartile range of product
# price (EUR). Right panel: within-market mean robust minimum delivery
# time (days), by FBA status.
# -----------------------------------------------------------------------------
struct_panel = df_final.groupby('market_order').agg(
    price_iqr=('prezzo', lambda s: float(np.percentile(s.dropna(), 75)
                                         - np.percentile(s.dropna(), 25))),
    price_med=('prezzo', 'median'),
).reset_index()

delivery_by_fba = df_final.groupby(['market_order', 'fba_from_shipper']).agg(
    mean_delivery=('g_cons_min_robust', 'mean'),
).reset_index()

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(11.0, 3.6))

# Left panel. Within-market price IQR per chronologically ordered
# market index.
ax_left.plot(struct_panel['market_order'], struct_panel['price_iqr'],
             color='black', linewidth=1.2, marker='o', markersize=3.0)
ax_left.set_xlabel('Market index (chronological order)', fontsize=9)
ax_left.set_ylabel('Within-market price IQR (EUR)', fontsize=9)
ax_left.set_ylim(bottom=4)
mean_iqr = struct_panel['price_iqr'].mean()
ax_left.axhline(mean_iqr, color='black', linestyle=':', linewidth=0.8)
ax_left.text(1, mean_iqr, f'  mean = {mean_iqr:.1f} EUR',
             fontsize=8, va='bottom')
ax_left.tick_params(direction='out', length=3)

# Right panel. Within-market mean minimum delivery time per market,
# by FBA status.
deliv_fba = delivery_by_fba[delivery_by_fba['fba_from_shipper'] == 1]
deliv_non = delivery_by_fba[delivery_by_fba['fba_from_shipper'] == 0]
ax_right.plot(deliv_fba['market_order'], deliv_fba['mean_delivery'],
              color=COLOR_FBA_LINE, linestyle='-', linewidth=1.2,
              marker='o', markersize=3.0,
              markerfacecolor=COLOR_FBA_FILL, label='FBA mean delivery (days)')
ax_right.plot(deliv_non['market_order'], deliv_non['mean_delivery'],
              color=COLOR_NONFBA_LINE, linestyle='--', linewidth=1.2,
              marker='o', markersize=3.0,
              markerfacecolor=COLOR_NONFBA_FILL, label='non-FBA mean delivery (days)')
ax_right.set_xlabel('Market index (chronological order)', fontsize=9)
ax_right.set_ylabel('Mean minimum delivery (days)', fontsize=9)
ax_right.set_ylim(bottom=4)
ax_right.legend(frameon=False, loc='upper right', fontsize=8)
ax_right.tick_params(direction='out', length=3)

fig.tight_layout()

f14_png = save_figure(fig, 'fig_F14_market_structure_stability')
print('Saved', f14_png)


---

## Figure F19. Added-variable plots for the S4 covariates

LaTeX label: `fig:avp-s4-covariates`. File:
`fig_F19_added_variable_plots_S4.pdf`. Rendered in appendix
Chapter `\ref{app:econometric-spec}`, Section
`\ref{app:static-specs}` of `main.tex`, immediately after
Figure `\ref{fig:fwl-fba-s4}`.

The six panels are Frisch-Waugh-Lovell added-variable plots for the
six non-FBA regressors of S4. Each panel residualises the listed
regressor on FBA and on the remaining five S4 covariates plus market
fixed effects, and plots the residualised `rank_pct` against the
residualised regressor. Reading the panels left to right, the
top-row slopes are log seller reviews $-0.0038$, positive-review
percentage $+0.0006$, and star rating $-0.0207$; the bottom-row
slopes are product price $+0.0306$, repaired shipping price
$+0.0303$, and standard delivery days $-0.0028$. Each slope equals
the corresponding coefficient in the S4 estimation reported in
Table `\ref{tab:ch5-static-models}`; standardized counterparts are
in Table `\ref{tab:ch5-standardized-features}`. The contact-courier
flag is omitted from the figure because its variation collapses
under residualisation; the table covers it.


In [ ]:
# -----------------------------------------------------------------------------
# Figure F19. Frisch-Waugh-Lovell added-variable plots for the six
# non-FBA S4 regressors. Each panel residualises the listed regressor
# on FBA and on the remaining five S4 covariates plus market fixed
# effects, then plots residualised rank_pct against the residualised
# regressor. The reported slope per panel equals the corresponding
# coefficient in the S4 estimation.
# -----------------------------------------------------------------------------
covariate_labels = {
    'log1p_num_valutazioni':       'log(1 + reviews)',
    'valutazioni_positive':        'Positive reviews (%)',
    'stelle':                      'Star rating',
    'prezzo':                      'Product price (EUR)',
    'prezzo_spedizione_repaired':  'Shipping price (EUR)',
    'g_cons_min_robust':           'Delivery, min days',
}
# contact_courier_flag is omitted because its residualised variation
# collapses heavily as a binary regressor; the table reports its
# coefficient.

avp_records = []
fig, axes = plt.subplots(2, 3, figsize=(11.0, 6.5))
axes = axes.ravel()

for ax, (col, label) in zip(axes, covariate_labels.items()):
    # Auxiliary regressions for the FWL construction: the target
    # covariate and rank_pct are regressed on FBA, the other five
    # S4 controls, and market fixed effects; the slope through the
    # two residuals equals the S4 coefficient on the target covariate.
    other_controls = [c for c in s4_controls if c != col]
    rhs = ' + '.join(['fba_from_shipper'] + other_controls + ['C(market_id)'])
    res_aux_x = smf.ols(f'{col} ~ {rhs}', data=df_final).fit()
    res_aux_y = smf.ols(f'rank_pct ~ {rhs}', data=df_final).fit()
    e_x_av = np.asarray(res_aux_x.resid, dtype=float)
    e_y_av = np.asarray(res_aux_y.resid, dtype=float)
    slope_av = float(np.sum(e_x_av * e_y_av) / np.sum(e_x_av ** 2))
    avp_records.append({'covariate': col, 'slope': slope_av})

    # Reference slope from the S4 fit, for cross-checking with the
    # FWL slope obtained from the auxiliary regressions.
    slope_s4 = float(res_s4.params[col])

    ax.scatter(e_x_av, e_y_av, s=3, alpha=0.20, color='gray', edgecolor='none')
    ax.axhline(0.0, color='black', linestyle=':', linewidth=0.5)
    ax.axvline(0.0, color='black', linestyle=':', linewidth=0.5)
    x_line = np.linspace(np.percentile(e_x_av, 1),
                         np.percentile(e_x_av, 99), 200)
    ax.plot(x_line, slope_av * x_line, color='black', linewidth=1.2,
            zorder=3)
    ax.set_xlabel(f'e({label})', fontsize=8)
    ax.set_ylabel('e(rank_pct)', fontsize=8)
    ax.set_title(f'slope = {slope_av:+.4f}', loc='left',
                 fontsize=9, pad=4)
    ax.tick_params(direction='out', length=3, labelsize=7)
    # Trim the axis range to the 1st-99th percentile of the
    # residualised variables to keep the scatter readable.
    p1, p99 = np.percentile(e_x_av, [1, 99])
    ax.set_xlim(p1, p99)
    py1, py99 = np.percentile(e_y_av, [1, 99])
    ax.set_ylim(py1, py99)

print(pd.DataFrame(avp_records).round(5).to_string(index=False))

fig.tight_layout()

f19_png = save_figure(fig, 'fig_F19_added_variable_plots_S4')
print('Saved', f19_png)


---

## Figure F20. Logistic prediction of FBA status from seller- and offer-level observables

LaTeX label: `fig:fba-prediction-model`. File:
`fig_F20_fba_prediction_model.pdf`. Rendered in appendix
Chapter `\ref{app:econometric-spec}`, Section
`\ref{app:static-specs}` of `main.tex`, after Figure
`\ref{fig:avp-s4-covariates}`.

The model is a logistic regression of `fba_from_shipper` on five
seller- and offer-level observables (log seller reviews,
positive-review percentage, star rating, product price, robust
minimum delivery days) on the full 5,107-row third-party panel.
Repaired shipping price and the contact-courier flag are excluded
from the model because they induce quasi-complete separation, since
FBA rows display zero repaired shipping by construction of the
displayed shipping component. The four panels report, clockwise
from top-left, the coefficient estimates with plus/minus 1.96
standard-error bars, the in-sample ROC curve with area under the
curve, the confusion matrix at the 0.50 classification threshold,
and in-sample calibration within deciles of the predicted
probability. The in-sample AUC is 0.916, not validated out of
sample; the misclassification share at the 0.50 threshold is
consistent with the propensity-score overlap pattern reported in
Figure `\ref{fig:propensity-density-by-fba}`.


In [ ]:
# -----------------------------------------------------------------------
# Figure F20. Logistic regression of FBA status on five seller- and
# offer-level observables, with four diagnostic panels (coefficients,
# ROC, calibration deciles, confusion matrix at the 0.50 threshold).
# Shipping price and contact-courier flag are excluded from the
# right-hand side because they induce quasi-complete separation
# (FBA rows display zero repaired shipping by construction).
# -----------------------------------------------------------------------
from sklearn.metrics import roc_curve, auc, confusion_matrix

# Right-hand side: five seller- and offer-level observables that vary
# independently of fulfillment mode. Shipping price and contact-courier
# flag are not used as regressors here because, on the FBA side, they
# are constant by construction and induce quasi-complete separation.
pred_vars = [
    'log1p_num_valutazioni',
    'valutazioni_positive',
    'stelle',
    'prezzo',
    'g_cons_min_robust',
]
pred_labels = {
    'log1p_num_valutazioni':  'log(1 + reviews)',
    'valutazioni_positive':   'Positive reviews (%)',
    'stelle':                 'Star rating',
    'prezzo':                 'Product price (EUR)',
    'g_cons_min_robust':      'Delivery, min days',
}
rhs_logit = ' + '.join(pred_vars)
res_logit = smf.logit(
    f'fba_from_shipper ~ {rhs_logit}', data=df_final
).fit(disp=0, maxiter=200)
print(res_logit.summary2().tables[1])

y_true = df_final['fba_from_shipper'].to_numpy().astype(int)
y_prob = res_logit.predict(df_final).to_numpy()
y_pred = (y_prob >= 0.50).astype(int)
fpr_r, tpr_r, _ = roc_curve(y_true, y_prob)
auc_score = auc(fpr_r, tpr_r)
cm = confusion_matrix(y_true, y_pred)
print(f'AUC={auc_score:.4f}  pseudo-R2={res_logit.prsquared:.4f}')

# In-sample calibration within deciles of the predicted probability.
decile_edges = np.quantile(y_prob, np.linspace(0, 1, 11))
decile_edges[-1] += 1e-9
cal_rows = []
for k in range(10):
    sel = (y_prob >= decile_edges[k]) & (y_prob < decile_edges[k + 1])
    n_k = int(sel.sum())
    cal_rows.append({
        'k': k, 'n': n_k,
        'prob_mean': float(np.mean(y_prob[sel])) if n_k else np.nan,
        'obs_rate':  float(np.mean(y_true[sel])) if n_k else np.nan,
    })
df_logit_cal = pd.DataFrame(cal_rows)

# 2x2 layout: coefficients, ROC, calibration, confusion matrix.
fig, axes2 = plt.subplots(2, 2, figsize=(11.0, 8.0))
ax_coef, ax_roc = axes2[0]
ax_cal,  ax_cm  = axes2[1]
fig.subplots_adjust(hspace=0.38, wspace=0.30)

# Panel 1. Coefficient estimates with plus/minus 1.96 standard-error
# bars.
coef_df = pd.DataFrame({
    'label': [pred_labels[v] for v in pred_vars],
    'coef':  res_logit.params[pred_vars].to_numpy(),
    'se':    res_logit.bse[pred_vars].to_numpy(),
}).sort_values('coef').reset_index(drop=True)
y_c = np.arange(len(coef_df))
colours_c = [COLOR_FBA_LINE if c > 0 else COLOR_NONFBA_LINE
             for c in coef_df['coef']]
ax_coef.barh(y_c, coef_df['coef'], height=0.55,
             color=colours_c, edgecolor='black', linewidth=0.5, alpha=0.85)
ax_coef.errorbar(coef_df['coef'], y_c,
                 xerr=1.96 * coef_df['se'],
                 fmt='none', color='black', linewidth=0.9, capsize=3)
ax_coef.axvline(0.0, color='black', linestyle=':', linewidth=0.7)
ax_coef.set_yticks(y_c)
ax_coef.set_yticklabels(coef_df['label'], fontsize=9)
ax_coef.set_xlabel('Logit coefficient  \u00b1 95% CI', fontsize=9)
ax_coef.set_title('Coefficient estimates', loc='left', fontsize=10)
ax_coef.tick_params(direction='out', length=3)

# Panel 2. In-sample ROC curve with AUC.
ax_roc.plot(fpr_r, tpr_r, color=COLOR_FBA_LINE, linewidth=1.6,
            label=f'AUC = {auc_score:.3f}')
ax_roc.plot([0, 1], [0, 1], color='black', linestyle=':', linewidth=0.7)
ax_roc.set_xlabel('False positive rate', fontsize=9)
ax_roc.set_ylabel('True positive rate', fontsize=9)
ax_roc.set_title('ROC curve (in-sample)', loc='left', fontsize=10)
ax_roc.legend(frameon=False, fontsize=9, loc='lower right')
ax_roc.tick_params(direction='out', length=3)

# Panel 3. Calibration of predicted probabilities within deciles of
# the predicted probability.
valid_cal = df_logit_cal.dropna()
ax_cal.scatter(valid_cal['prob_mean'], valid_cal['obs_rate'],
               s=65, color=COLOR_FBA_FILL, edgecolor=COLOR_FBA_LINE,
               linewidth=0.9, zorder=3)
ax_cal.plot([0, 1], [0, 1], color='black', linestyle='-', linewidth=0.8)
for _, row in valid_cal.iterrows():
    ax_cal.annotate(f"{int(row['k'])+1}",
                    (row['prob_mean'], row['obs_rate']),
                    xytext=(5, 0), textcoords='offset points',
                    fontsize=7.5, color=COLOR_FBA_LINE)
ax_cal.set_xlabel('Mean predicted probability within decile', fontsize=9)
ax_cal.set_ylabel('Observed FBA rate within decile', fontsize=9)
ax_cal.set_title('Calibration (predicted vs observed)', loc='left', fontsize=10)
ax_cal.set_xlim(0, 1); ax_cal.set_ylim(0, 1)
ax_cal.tick_params(direction='out', length=3)

# Panel 4. Confusion matrix at the 0.50 classification threshold.
group_names  = [['TN', 'FP'], ['FN', 'TP']]
n_tot = cm.sum()
im_cm = ax_cm.imshow(cm, interpolation='nearest',
                     cmap=plt.cm.Oranges, aspect='auto')
ax_cm.set_xticks([0, 1])
ax_cm.set_yticks([0, 1])
ax_cm.set_xticklabels(['non-FBA\n(pred.)', 'FBA\n(pred.)'], fontsize=9)
ax_cm.set_yticklabels(['non-FBA\n(true)', 'FBA\n(true)'], fontsize=9)
ax_cm.set_title('Confusion matrix (threshold = 0.50)', loc='left', fontsize=10)
thresh_cm = im_cm.norm(cm.max()) / 2.0
for i in range(2):
    for j in range(2):
        c_txt = 'white' if im_cm.norm(cm[i, j]) > thresh_cm else 'black'
        lbl = f'{group_names[i][j]}\n{cm[i,j]}\n({cm[i,j]/n_tot:.1%})'
        ax_cm.text(j, i, lbl, ha='center', va='center',
                   fontsize=11, color=c_txt)

f20_png = save_figure(fig, 'fig_F20_fba_prediction_model')
print('Saved', f20_png)


---

## Figure FA. Attenuation of the FBA coefficient across nested static specifications

LaTeX label: `fig:attenuation-decomposition`. File:
`fig_FA_attenuation_decomposition_S1_S4.pdf`. Rendered in Chapter
`\ref{ch:results}`, Section `\ref{sec:results-attenuation}` of
`main.tex`, immediately after Table `\ref{tab:ch5-attenuation}`.

The left panel plots the FBA coefficient on `rank_pct` across the
four nested static specifications S1 to S4 of
Table `\ref{tab:ch5-static-models}`, with shaded plus/minus 1.96
standard-error bands. The right panel decomposes the movement of
the FBA coefficient across the three control blocks, in coefficient
units and as a percentage of the S1 baseline of $-0.380$. The
reputation block ($S1 \to S2$) enlarges the absolute coefficient
by 4 percent of the S1 baseline; the price-and-delivery block
($S2 \to S3$) shifts it by +91 percent; the split-price refinement
($S3 \to S4$) contributes less than 1 percent. The percentages are
scaled by the S1 baseline and do not sum to 100 percent because the
reputation block moves the coefficient in the opposite direction.
The decomposition is the descriptive coefficient accounting of
Section `\ref{sec:results-attenuation}` and is not a causal
mediation share.


In [ ]:
# -----------------------------------------------------------------------
# Figure FA. FBA coefficient across the four nested static
# specifications S1 to S4 (left panel) and decomposition of the
# movement across the three control blocks, expressed in coefficient
# units and as a percentage of the S1 baseline of approximately
# -0.380 (right panel). The two-way OLS standard-error bands shown
# in the left panel are for visual orientation; the clustered-SE
# inference is in Table \ref{tab:ch5-static-models}.
# -----------------------------------------------------------------------
import statsmodels.formula.api as smf

specs_fa = [
    ('S1', 'rank_pct ~ fba_from_shipper + C(market_id)'),
    ('S2', 'rank_pct ~ fba_from_shipper + log1p_num_valutazioni + valutazioni_positive + stelle + C(market_id)'),
    ('S3', 'rank_pct ~ fba_from_shipper + log1p_num_valutazioni + valutazioni_positive + stelle + prezzo_totale_reconstructed + contact_courier_flag + g_cons_min_robust + C(market_id)'),
    ('S4', 'rank_pct ~ fba_from_shipper + log1p_num_valutazioni + valutazioni_positive + stelle + prezzo + prezzo_spedizione_repaired + contact_courier_flag + g_cons_min_robust + C(market_id)'),
]
res_fa = []
for label, f in specs_fa:
    r = smf.ols(f, data=df_final).fit()
    res_fa.append({'label': label, 'coef': float(r.params['fba_from_shipper']),
                   'se': float(r.bse['fba_from_shipper'])})
    print(f"{label}: FBA = {res_fa[-1]['coef']:+.4f} (SE={res_fa[-1]['se']:.4f})")

coefs_fa = [r['coef'] for r in res_fa]
ses_fa   = [r['se']   for r in res_fa]
block_labels_fa = ['S1\nFBA + market FE\n(no controls)', 'S2\n+ reputation\n(reviews, stars)', 'S3\n+ total price\n+ delivery', 'S4\n+ split price\n(product/ship)']

fig_fa, (ax_fa1, ax_fa2) = plt.subplots(1, 2, figsize=(12, 4.5))

xs_fa = np.arange(4)
ax_fa1.fill_between(xs_fa,
                    np.array(coefs_fa) - 1.96*np.array(ses_fa),
                    np.array(coefs_fa) + 1.96*np.array(ses_fa),
                    color='black', alpha=0.10, linewidth=0)
ax_fa1.plot(xs_fa, coefs_fa, color='black', linewidth=1.6, zorder=2)
ax_fa1.scatter(xs_fa, coefs_fa, s=70, color='black', zorder=3)
ax_fa1.axhline(0, color='black', linestyle=':', linewidth=0.8)
for xi, r in zip(xs_fa, res_fa):
    if xi == 0:
        # S1 label is placed above the point.
        xoff, yoff, ha_ = 0, 10, 'center'
    elif xi == 1:
        # S2 label is placed to the right of the point to avoid the
        # confidence band.
        xoff, yoff, ha_ = 14, 0, 'left'
    else:
        # S3, S4 labels are placed above the point.
        xoff, yoff, ha_ = 0, 10, 'center'
    ax_fa1.annotate(f"{r['coef']:+.4f}", (xi, r['coef']),
                    xytext=(xoff, yoff), textcoords='offset points',
                    ha=ha_, fontsize=9.5)
ax_fa1.set_xticks(xs_fa)
ax_fa1.set_xticklabels(block_labels_fa, fontsize=8.5)
ax_fa1.set_ylabel('FBA coefficient on rank_pct  (\u00b1 95% CI shaded)', fontsize=9)
ax_fa1.set_title('Attenuation across nested specifications', loc='left', fontsize=10)
ax_fa1.axhspan(coefs_fa[-1], coefs_fa[0], alpha=0.06, color=COLOR_NONFBA_FILL, linewidth=0)
ax_fa1.tick_params(direction='out', length=3)

diffs_fa   = [coefs_fa[i+1]-coefs_fa[i] for i in range(3)]
pcts_fa    = [abs(d)/abs(coefs_fa[0])*100 for d in diffs_fa]
bc_labels  = ['Reputation\n(S1\u2192S2)', 'Price + delivery\n(S2\u2192S3)', 'Split-price\n(S3\u2192S4)']
bc_colours = [COLOR_NONFBA_LINE, COLOR_FBA_LINE, '#7B4F8A']

ax_fa2.bar(np.arange(3), diffs_fa, color=bc_colours,
           edgecolor='black', linewidth=0.7, width=0.55, alpha=0.88)
ax_fa2.axhline(0, color='black', linestyle=':', linewidth=0.7)
for xi, (d, pct) in enumerate(zip(diffs_fa, pcts_fa)):
    if xi == 1:
        # The S2->S3 bar carries +91% of the S1 baseline; place the
        # label inside the bar at half-height.
        ax_fa2.text(xi, d/2, f'{d:+.3f}\n({pct:.0f}% of S1)',
                    ha='center', va='center', fontsize=9, color='black', fontweight='bold')
    else:
        # The reputation (S1->S2) and split-price (S3->S4) bars are
        # small; place the label just above the zero axis, outside
        # the bar.
        ax_fa2.text(xi, 0.004, f'{d:+.3f}\n({pct:.0f}% of S1)',
                    ha='center', va='bottom', fontsize=9, color='black', fontweight='bold')
ax_fa2.set_xticks(np.arange(3))
ax_fa2.set_xticklabels(bc_labels, fontsize=9)
ax_fa2.set_ylabel('Change in FBA coefficient', fontsize=9)
ax_fa2.set_title(f'Control-block contributions\n(S1 baseline = {coefs_fa[0]:+.3f})',
                 loc='left', fontsize=10)
ax_fa2.tick_params(direction='out', length=3)

fig_fa.tight_layout()
fA_png = save_figure(fig_fa, 'fig_FA_attenuation_decomposition_S1_S4')
print('Saved', fA_png)


---

## Figure FB. Progressive demeaning of the FBA rank gap across the nested specifications

LaTeX label: `fig:within-market-demeaned-gap`. File:
`fig_FB_within_market_demeaned_gap.pdf`. Rendered in the
supplementary chapter `\ref{app:additional-results}`, subsection
*Static interaction terms*.

The left panel reports the group mean of `rank_pct` for FBA and
non-FBA rows at four demeaning steps: (i) raw group means; (ii)
residuals after market-only demeaning at the S1 level; (iii)
residuals after market plus reputation demeaning at the S2 level;
(iv) residuals after the full S4 demeaning. The right panel reports
the corresponding FBA minus non-FBA gap with 95 percent confidence
intervals based on ordinary group standard errors. The gap at the
full S4 step is $-0.031$, which is the group-mean counterpart of the
S4 coefficient of $-0.051$ in Table `\ref{tab:ch5-static-models}`.
The OLS S4 coefficient differs numerically from the step (iv) gap
because the OLS coefficient is a Frisch-Waugh-Lovell slope (the
covariance between residualised `rank_pct` and residualised FBA
divided by the variance of residualised FBA), while the step (iv)
gap is the difference of unweighted group means of the residualised
outcome. The two quantities share the same sign and identifying
variation.


In [ ]:
# -----------------------------------------------------------------------
# Figure FB. Group means of rank_pct for FBA and non-FBA rows at four
# demeaning steps (raw, market FE, market+reputation FE, full S4),
# and the corresponding group-mean FBA minus non-FBA gap. The
# group-mean gap at the full S4 step is the unweighted-mean
# counterpart of the S4 OLS coefficient.
# -----------------------------------------------------------------------
import statsmodels.formula.api as smf

formula_s4_controls_only = (
    'rank_pct ~ log1p_num_valutazioni + valutazioni_positive + stelle + '
    'prezzo + prezzo_spedizione_repaired + contact_courier_flag + '
    'g_cons_min_robust + C(market_id)'
)

resid_raw  = df_final['rank_pct'].to_numpy(dtype=float)
resid_mkt  = smf.ols('rank_pct ~ C(market_id)', data=df_final).fit().resid.to_numpy()
resid_rep  = smf.ols('rank_pct ~ log1p_num_valutazioni + valutazioni_positive + stelle + C(market_id)',
                     data=df_final).fit().resid.to_numpy()
resid_s4r  = smf.ols(formula_s4_controls_only, data=df_final).fit().resid.to_numpy()

fba_mask = df_final['fba_from_shipper'].to_numpy() == 1
step_labels_fb = ['Raw\n(no demeaning)', 'Within-market\n(S1 level)', '+Reputation\n(S2 level)', '+All S4\ncontrols']
step_data = [resid_raw, resid_mkt, resid_rep, resid_s4r]

gap_rows = []
for arr in step_data:
    fba_  = arr[fba_mask]
    non_  = arr[~fba_mask]
    mf, sf = fba_.mean(), fba_.std(ddof=1)/np.sqrt(len(fba_))
    mn, sn = non_.mean(), non_.std(ddof=1)/np.sqrt(len(non_))
    gap_rows.append({'fba_mean':mf,'fba_se':sf,'non_mean':mn,'non_se':sn,
                     'gap':mf-mn,'gap_se':np.sqrt(sf**2+sn**2)})

from matplotlib.lines import Line2D
fig_fb, (ax_fb1, ax_fb2) = plt.subplots(1, 2, figsize=(11, 4.5))
xs_fb = np.arange(4)
offset_fb = 0.12
for xi, gd in zip(xs_fb, gap_rows):
    ax_fb1.errorbar(xi-offset_fb, gd['fba_mean'], yerr=1.96*gd['fba_se'],
                    fmt='o', color=COLOR_FBA_LINE, markerfacecolor=COLOR_FBA_FILL,
                    markersize=7, linewidth=0, capsize=3, elinewidth=1.0)
    ax_fb1.errorbar(xi+offset_fb, gd['non_mean'], yerr=1.96*gd['non_se'],
                    fmt='o', color=COLOR_NONFBA_LINE, markerfacecolor=COLOR_NONFBA_FILL,
                    markersize=7, linewidth=0, capsize=3, elinewidth=1.0)
ax_fb1.plot(xs_fb-offset_fb, [gd['fba_mean'] for gd in gap_rows],
            color=COLOR_FBA_LINE, linestyle='-', linewidth=1.3)
ax_fb1.plot(xs_fb+offset_fb, [gd['non_mean'] for gd in gap_rows],
            color=COLOR_NONFBA_LINE, linestyle='--', linewidth=1.3)
ax_fb1.axhline(0, color='black', linestyle=':', linewidth=0.7)
ax_fb1.set_xticks(xs_fb); ax_fb1.set_xticklabels(step_labels_fb, fontsize=9)
ax_fb1.set_ylabel('Group mean of (demeaned) rank_pct  \u00b1 95% CI', fontsize=9)
ax_fb1.set_title('FBA vs non-FBA rank_pct at each demeaning step', loc='left', fontsize=10)
leg_fb = [Line2D([0],[0],color=COLOR_FBA_LINE,linestyle='-',marker='o',
                 markerfacecolor=COLOR_FBA_FILL,markersize=6,linewidth=1.2,label='FBA'),
          Line2D([0],[0],color=COLOR_NONFBA_LINE,linestyle='--',marker='o',
                 markerfacecolor=COLOR_NONFBA_FILL,markersize=6,linewidth=1.2,label='non-FBA')]
ax_fb1.legend(handles=leg_fb, frameon=False, loc='upper right', fontsize=9)
ax_fb1.tick_params(direction='out', length=3)

ax_fb2.errorbar(xs_fb, [gd['gap'] for gd in gap_rows],
                yerr=[1.96*gd['gap_se'] for gd in gap_rows],
                fmt='o', color='black', markerfacecolor='black',
                markersize=7, linewidth=0, capsize=3, elinewidth=1.0, zorder=3)
ax_fb2.plot(xs_fb, [gd['gap'] for gd in gap_rows], color='black', linewidth=1.3)
ax_fb2.axhline(0, color='black', linestyle=':', linewidth=0.7)
for xi, gd in zip(xs_fb, gap_rows):
    if xi == 2:
        # The S2-level label is placed to the right of the point to
        # avoid overlap with the connecting line.
        ax_fb2.annotate(f"{gd['gap']:+.3f}", (xi, gd['gap']),
                        xytext=(8, 0), textcoords='offset points',
                        ha='left', va='center', fontsize=9)
    else:
        ax_fb2.annotate(f"{gd['gap']:+.3f}", (xi, gd['gap']),
                        xytext=(0, 9), textcoords='offset points',
                        ha='center', fontsize=9)
ax_fb2.set_xticks(xs_fb); ax_fb2.set_xticklabels(step_labels_fb, fontsize=9)
ax_fb2.set_ylabel('FBA \u2212 non-FBA gap  \u00b1 95% CI', fontsize=9)
ax_fb2.set_title('Residual rank gap\n(= FBA coefficient at that demeaning level)', loc='left', fontsize=10)
ax_fb2.tick_params(direction='out', length=3)

fig_fb.tight_layout()
fB_png = save_figure(fig_fb, 'fig_FB_within_market_demeaned_gap')
print('Saved', fB_png)


---

## Figure FC. Per-market FBA rank advantage across all 62 retained markets

LaTeX label: `fig:per-market-fba-gap`. File:
`fig_FC_per_market_fba_gap.pdf`. Rendered in the supplementary
chapter `\ref{app:additional-results}`, subsection
*Static interaction terms*, after Figure
`\ref{fig:within-market-demeaned-gap}`.

The left panel reports the per-market mean `rank_pct` for FBA (dots)
and non-FBA (squares), connected by a vertical segment, with markets
ordered by the FBA per-market mean. The right panel reports the
empirical density of the per-market FBA minus non-FBA mean `rank_pct`
gap; the dashed line marks the panel-wide mean gap of $-0.380$, which
coincides with the S1 coefficient in Table
`\ref{tab:ch5-static-models}`. The per-market gap is strictly
negative in all 62 retained markets; the right-panel density support
is entirely negative. The within-market FE design of Chapter
`\ref{ch:methodology}` is therefore not extrapolating from a subset
of markets where the gap happens to be large.


In [ ]:
# -----------------------------------------------------------------------
# Figure FC. Per-market mean rank_pct for FBA (dots) and non-FBA
# (squares), connected by a vertical segment, with markets sorted by
# the FBA per-market mean (left panel); and the empirical density of
# the per-market FBA minus non-FBA gap (right panel). The panel-wide
# mean gap equals the S1 coefficient.
# -----------------------------------------------------------------------
from scipy.stats import gaussian_kde

mkt_fc = (
    df_final.groupby(['market_id', 'fba_from_shipper'])['rank_pct']
    .mean().reset_index()
)
mkt_fba_fc = mkt_fc[mkt_fc['fba_from_shipper']==1].set_index('market_id')['rank_pct']
mkt_non_fc = mkt_fc[mkt_fc['fba_from_shipper']==0].set_index('market_id')['rank_pct']
mkt_joint   = mkt_fba_fc.rename('fba').to_frame().join(
                  mkt_non_fc.rename('non'), how='inner')
mkt_joint['gap'] = mkt_joint['fba'] - mkt_joint['non']
mkt_joint = mkt_joint.sort_values('fba').reset_index()

gap_vals_fc = mkt_joint['gap'].to_numpy()
print(f"FC: {(gap_vals_fc<0).mean():.0%} of markets FBA ranks better, "
      f"mean gap={gap_vals_fc.mean():+.3f}")

fig_fc, (ax_fc1, ax_fc2) = plt.subplots(1, 2, figsize=(11, 4.2))
xs_fc = np.arange(len(mkt_joint))

for xi, row in zip(xs_fc, mkt_joint.itertuples()):
    col = COLOR_FBA_LINE if row.gap < 0 else COLOR_NONFBA_LINE
    ax_fc1.plot([xi,xi],[row.fba,row.non], color=col, linewidth=0.8, alpha=0.55)
ax_fc1.scatter(xs_fc, mkt_joint['fba'], s=18, color=COLOR_FBA_LINE, zorder=3, alpha=0.85)
ax_fc1.scatter(xs_fc, mkt_joint['non'], s=18, color=COLOR_NONFBA_LINE,
               marker='s', zorder=3, alpha=0.85)
ax_fc1.axhline(df_final.loc[df_final['fba_from_shipper']==1,'rank_pct'].mean(),
               color=COLOR_FBA_LINE,  linestyle='--', linewidth=0.8, alpha=0.5)
ax_fc1.axhline(df_final.loc[df_final['fba_from_shipper']==0,'rank_pct'].mean(),
               color=COLOR_NONFBA_LINE, linestyle='--', linewidth=0.8, alpha=0.5)

from matplotlib.lines import Line2D
leg_fc = [Line2D([0],[0],color=COLOR_FBA_LINE,linestyle='none',marker='o',
                 markersize=6,label='FBA mean'),
          Line2D([0],[0],color=COLOR_NONFBA_LINE,linestyle='none',marker='s',
                 markersize=6,label='non-FBA mean')]
ax_fc1.legend(handles=leg_fc, frameon=False, fontsize=9)
ax_fc1.set_xlim(-1, len(mkt_joint))
ax_fc1.set_ylim(0,1)
ax_fc1.set_xlabel('Markets sorted by FBA mean rank_pct (62 markets)', fontsize=9)
ax_fc1.set_ylabel('Mean within-market rank_pct', fontsize=9)
ax_fc1.set_title('Per-market FBA vs non-FBA mean rank_pct', loc='left', fontsize=10)
ax_fc1.tick_params(direction='out', length=3)

grid_fc = np.linspace(gap_vals_fc.min()-0.02, gap_vals_fc.max()+0.02, 200)
kde_fc = gaussian_kde(gap_vals_fc, bw_method='scott')(grid_fc)
ax_fc2.fill_between(grid_fc[grid_fc<0], 0, kde_fc[grid_fc<0],
                    color=COLOR_FBA_FILL, alpha=0.80, linewidth=0)
ax_fc2.plot(grid_fc, kde_fc, color='black', linewidth=1.4)
ax_fc2.axvline(0, color='black', linestyle=':', linewidth=0.8)
ax_fc2.axvline(gap_vals_fc.mean(), color='black', linestyle='--', linewidth=1.0)
ax_fc2.set_xlabel('FBA \u2212 non-FBA mean rank_pct gap', fontsize=9)
ax_fc2.set_ylabel('Empirical density (62 markets)', fontsize=9)
ax_fc2.set_title('Distribution of per-market FBA rank advantage', loc='left', fontsize=10)
ax_fc2.set_ylim(bottom=0)
ax_fc2.tick_params(direction='out', length=3)

# Legend reporting the share of markets in which FBA ranks better
# (here 100%) and the panel-wide mean gap.
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
leg_fc2 = [
    Patch(facecolor=COLOR_FBA_FILL, alpha=0.80, edgecolor='none',
          label='100% of markets: FBA ranks better'),
    Line2D([0], [0], color='black', linestyle='--', linewidth=1.0,
           label=f'mean gap = {gap_vals_fc.mean():+.3f}'),
]
ax_fc2.legend(handles=leg_fc2, frameon=False, loc='upper right',
              fontsize=8.5, handlelength=2.0)

fig_fc.tight_layout()
fC_png = save_figure(fig_fc, 'fig_FC_per_market_fba_gap')
print('Saved', fC_png)


---

## Figure FD. Contemporaneous D1 and lead-dropout placebo

LaTeX label: `fig:d1-placebo`. File:
`fig_FD_d1_placebo_test.pdf`. Rendered in Chapter
`\ref{ch:robustness}`, Section
`\ref{sec:robustness-dynamic-placebo}` of `main.tex`, alongside the
temporal-placebo discussion that surrounds
Table `\ref{tab:app-ch6-dynamic-placebos}`.

The left panel reports per-level binned means of the FBA minus
non-FBA difference in `rank_pct_improvement` with 95 percent
confidence intervals, against contemporaneous `dropouts_total_focal`.
The right panel reports the same quantity against lead dropouts in
the following transition; lead-dropout rows with no $t{+}1$
transition are excluded so the two panels are estimated on the same
sample. The contemporaneous panel displays a sign-positive pattern
over the upper exposure range; the lead-dropout panel is
approximately flat around zero. The in-figure coefficients are from
bivariate regressions without seller or transition fixed effects and
are reported as a visual diagnostic; the table-level placebo
coefficient ($+0.000063$, $p=0.885$) and the fixed-effects D1
coefficient ($+0.002467$, $p=0.0020$) are in
Table `\ref{tab:app-ch6-dynamic-placebos}` and
Table `\ref{tab:ch5-dynamic-models}` respectively.


In [ ]:
# -----------------------------------------------------------------------
# Figure FD. Per-level binned FBA minus non-FBA difference in
# rank_pct_improvement, against contemporaneous dropouts_total_focal
# (left panel) and against lead dropouts in the t+1 transition
# (right panel). The figure is the visual diagnostic that accompanies
# the lead-dropout placebo of Section
# \ref{sec:robustness-dynamic-placebo}.
# -----------------------------------------------------------------------
import statsmodels.formula.api as smf

# Rebuild the transition panel and attach the one-step-ahead lead
# dropouts to each transition.
pivot_rank_d = df_final.pivot_table(index='market_order', columns='seller_id',
                                     values='rank_pct')
pivot_fba_d  = df_final.pivot_table(index='market_order', columns='seller_id',
                                     values='fba_from_shipper')
orders_d = sorted(pivot_rank_d.index)

trans_d = []
for prev_o, next_o in zip(orders_d[:-1], orders_d[1:]):
    rp = pivot_rank_d.loc[prev_o]; rn = pivot_rank_d.loc[next_o]
    fp = pivot_fba_d.loc[prev_o]
    in_p, in_n = set(rp.dropna().index), set(rn.dropna().index)
    n_drop = len(in_p - in_n)
    for s in in_p & in_n:
        trans_d.append({'seller_id':s,'m_t':int(prev_o),'m_tp1':int(next_o),
                        'improvement':float(rp[s])-float(rn[s]),
                        'fba_t':int(fp[s]),'dropouts_total':int(n_drop)})

df_td = pd.DataFrame(trans_d)

# Lead dropouts: number of disappearing sellers in the next
# transition (m_tp1 -> m_tp1+1).
drop_map_d = df_td.groupby(['m_t','m_tp1'])['dropouts_total'].first().to_dict()
def get_lead(row):
    idx_tp1 = orders_d.index(row['m_tp1'])
    if idx_tp1+1 < len(orders_d):
        return drop_map_d.get((row['m_tp1'], orders_d[idx_tp1+1]), np.nan)
    return np.nan

df_td['dropouts_lead'] = df_td.apply(get_lead, axis=1)
df_td_lead = df_td.dropna(subset=['dropouts_lead']).copy()
df_td_lead['fba_x_drop_lag']  = df_td_lead['fba_t']*df_td_lead['dropouts_total']
df_td_lead['fba_x_drop_lead'] = df_td_lead['fba_t']*df_td_lead['dropouts_lead']

res_d1s = smf.ols('improvement ~ fba_x_drop_lag  + C(seller_id) + C(m_t)',
                  data=df_td_lead).fit()
res_plc = smf.ols('improvement ~ fba_x_drop_lead + C(seller_id) + C(m_t)',
                  data=df_td_lead).fit()
d1s_c = float(res_d1s.params.get('fba_x_drop_lag', np.nan))
d1s_p = float(res_d1s.pvalues.get('fba_x_drop_lag', np.nan))
plc_c = float(res_plc.params.get('fba_x_drop_lead', np.nan))
plc_p = float(res_plc.pvalues.get('fba_x_drop_lead', np.nan))
print(f"D1 sub: {d1s_c:+.5f}  p={d1s_p:.3f}")
print(f"Placebo: {plc_c:+.5f}  p={plc_p:.3f}")

def _binned_diff(df_, x_col, y_col, g_col):
    rows = []
    for k in sorted(df_[x_col].unique()):
        f_ = df_.loc[(df_[x_col]==k)&(df_[g_col]==1), y_col]
        n_ = df_.loc[(df_[x_col]==k)&(df_[g_col]==0), y_col]
        if len(f_)>2 and len(n_)>2:
            d = f_.mean()-n_.mean()
            se= np.sqrt(f_.std(ddof=1)**2/len(f_) + n_.std(ddof=1)**2/len(n_))
            rows.append({'k':k,'diff':d,'se':se,'n':len(f_)+len(n_)})
    return pd.DataFrame(rows)

bg_c = _binned_diff(df_td_lead,'dropouts_total','improvement','fba_t')
bg_p = _binned_diff(df_td_lead,'dropouts_lead', 'improvement','fba_t')

fig_fd, axes_fd = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, bg, title, subt in [
    (axes_fd[0], bg_c,
     'Contemporaneous D1 (actual)',
     f'coef = {d1s_c:+.5f}  (p = {d1s_p:.3f})'),
    (axes_fd[1], bg_p,
     'Lead-dropout placebo (falsification)',
     f'coef = {plc_c:+.5f}  (p = {plc_p:.3f})'),
]:
    ax.errorbar(bg['k'], bg['diff'], yerr=1.96*bg['se'],
                fmt='o', color='black', markerfacecolor='black',
                markersize=6, linewidth=0, capsize=3, elinewidth=1.0)
    ax.plot(bg['k'], bg['diff'], color='black', linewidth=1.2)
    ax.axhline(0, color='black', linestyle=':', linewidth=0.8)
    for _, row in bg.iterrows():
        # Sample-size annotation placed just below the lower
        # error-bar cap of each per-level point.
        y_bottom = row['diff'] - 1.96 * row['se']
        ax.text(row['k'], y_bottom - 0.0004,
                f"n={int(row['n'])}", ha='center', va='top', fontsize=6.5, color='gray')
    ax.set_xlabel('dropouts_total_focal (sellers per transition)', fontsize=9)
    ax.set_title(f'{title}\n{subt}', loc='left', fontsize=9.5)
    ax.tick_params(direction='out', length=3)

axes_fd[0].set_ylabel('Mean rank_pct improvement: FBA \u2212 non-FBA  \u00b1 95% CI', fontsize=9)
# No global in-figure title: the LaTeX caption is the global
# descriptive layer. Panel headings are retained to distinguish the
# contemporaneous diagnostic from the placebo.
fig_fd.tight_layout()
fD_png = save_figure(fig_fd, 'fig_FD_d1_placebo_test')
print('Saved', fD_png)


---

## Figure FE. Seller tenure in the panel by FBA status

LaTeX label: `fig:seller-tenure-by-fba`. File:
`fig_FE_seller_tenure_panel.pdf`. Rendered in the supplementary
chapter `\ref{app:additional-results}`, subsection
*Dynamic power and rank-tier support*, before Figure
`\ref{fig:fba-share-by-rank-tier}`.

The left panel reports the distribution of the number of markets in
which each seller is observed, separately for FBA ($n=28$) and
non-FBA ($n=91$) seller identities, with per-group median guides.
The right panel reports the empirical survival curve, that is, the
fraction of sellers in each group present in at least $k$ markets.
The median tenure is about 39 markets ($38.5$ on the 28 FBA seller
identities) for FBA sellers and 60 markets for non-FBA sellers; the
in-figure legend rounds the FBA median down to 38 by integer
formatting. The tenure differential is relevant for the
survival-bias considerations in Section
`\ref{sec:robustness-serial-dependence}` and the identification
limits in Section `\ref{sec:method-identification-limits}`.


In [ ]:
# -----------------------------------------------------------------------
# Figure FE. Seller tenure in the panel by FBA status. Left panel:
# distribution of the number of markets in which each seller appears.
# Right panel: empirical survival curve, that is, the fraction of
# sellers present in at least k markets. Sellers are classified by
# the modal value of fba_from_shipper across their seller-market
# observations.
# -----------------------------------------------------------------------
seller_fe = (
    df_final.groupby('seller_id')
    .agg(n_markets=('market_id','nunique'),
         fba_sum=('fba_from_shipper','sum'),
         n_obs=('fba_from_shipper','count'))
    .reset_index()
)
seller_fe['fba_modal'] = (seller_fe['fba_sum']/seller_fe['n_obs'] >= 0.5).astype(int)
ten_fba = seller_fe.loc[seller_fe['fba_modal']==1,'n_markets'].to_numpy()
ten_non = seller_fe.loc[seller_fe['fba_modal']==0,'n_markets'].to_numpy()
n_mkts_fe = int(df_final['market_order'].nunique())
print(f"FBA sellers n={len(ten_fba)}, median tenure={np.median(ten_fba):.0f}")
print(f"non-FBA sellers n={len(ten_non)}, median tenure={np.median(ten_non):.0f}")

max_t = int(max(ten_fba.max(), ten_non.max()))
bins_fe = np.arange(0.5, max_t+1.5)
ctr_fe  = np.arange(1, max_t+1)
hf, _ = np.histogram(ten_fba, bins=bins_fe, density=True)
hn, _ = np.histogram(ten_non, bins=bins_fe, density=True)

from matplotlib.lines import Line2D
fig_fe, (ax_fe1, ax_fe2) = plt.subplots(1, 2, figsize=(11, 4.2))
bw = 0.42
ax_fe1.bar(ctr_fe-bw/2, hf, width=bw, color=COLOR_FBA_FILL, edgecolor=COLOR_FBA_LINE,
           linewidth=0.6, alpha=0.85, label=f'FBA sellers (n={len(ten_fba)})')
ax_fe1.bar(ctr_fe+bw/2, hn, width=bw, color=COLOR_NONFBA_FILL, edgecolor=COLOR_NONFBA_LINE,
           linewidth=0.6, alpha=0.85, label=f'non-FBA sellers (n={len(ten_non)})')
ax_fe1.axvline(np.median(ten_fba), color=COLOR_FBA_LINE, linestyle='--', linewidth=1.0)
ax_fe1.axvline(np.median(ten_non), color=COLOR_NONFBA_LINE, linestyle='--', linewidth=1.0)
ax_fe1.set_xlabel('Number of markets seller appears in (out of 62)', fontsize=9)
ax_fe1.set_ylabel('Empirical density', fontsize=9)
ax_fe1.set_title('Seller panel tenure by FBA status', loc='left', fontsize=10)
ax_fe1.legend(frameon=False, fontsize=9)
ax_fe1.tick_params(direction='out', length=3)

surv_f = np.array([(ten_fba>=k).mean() for k in range(1,n_mkts_fe+1)])
surv_n = np.array([(ten_non>=k).mean() for k in range(1,n_mkts_fe+1)])
xs_s = np.arange(1, n_mkts_fe+1)
ax_fe2.fill_between(xs_s, surv_f, color=COLOR_FBA_FILL,  alpha=0.55, linewidth=0)
ax_fe2.fill_between(xs_s, surv_n, color=COLOR_NONFBA_FILL,alpha=0.55, linewidth=0)
ax_fe2.plot(xs_s, surv_f, color=COLOR_FBA_LINE,  linewidth=1.6, linestyle='-')
ax_fe2.plot(xs_s, surv_n, color=COLOR_NONFBA_LINE,linewidth=1.6, linestyle='--')
ax_fe2.axhline(0.5, color='black', linestyle=':', linewidth=0.7)
leg_fe = [Line2D([0],[0],color=COLOR_FBA_LINE,linestyle='-',linewidth=1.6,
                 label=f'FBA (median={int(np.median(ten_fba))} mkts)'),
          Line2D([0],[0],color=COLOR_NONFBA_LINE,linestyle='--',linewidth=1.6,
                 label=f'non-FBA (median={int(np.median(ten_non))} mkts)')]
ax_fe2.legend(handles=leg_fe, frameon=False, fontsize=9)
ax_fe2.set_xlabel('Number of markets k', fontsize=9)
ax_fe2.set_ylabel('Fraction of sellers present in \u2265 k markets', fontsize=9)
ax_fe2.set_title('Panel survival curve by FBA status', loc='left', fontsize=10)
ax_fe2.set_xlim(1, n_mkts_fe); ax_fe2.set_ylim(0, 1)
ax_fe2.tick_params(direction='out', length=3)

fig_fe.tight_layout()
fE_png = save_figure(fig_fe, 'fig_FE_seller_tenure_panel')
print('Saved', fE_png)


---

## Figure FF. Per-market common-support coverage

LaTeX label: `fig:propensity-coverage-by-market`. File:
`fig_FF_propensity_overlap_by_market.pdf`. Rendered in the
supplementary chapter `\ref{app:additional-results}`, subsection
*Overlap and balance*, after Figure
`\ref{fig:smd-before-after-trim}`.

The left panel reports the per-market share of sellers whose
estimated propensity score falls inside the common-support range
$[0.056,\,0.975]$, by chronological market index, with a horizontal
reference at the panel-wide mean coverage of approximately 42.8
percent. The right panel reports per-market coverage against the
within-market FBA share, with marker colour encoding the within-market
seller count. Per-market coverage ranges from approximately 36 to 50
percent and is mildly increasing in the within-market FBA share. The
residual-gap collapse inside the trimmed sample reported in
Table `\ref{tab:app-ch6-common-support-residual-gap}` is therefore
not driven by a subset of markets with anomalous coverage.


In [ ]:
# -----------------------------------------------------------------------
# Figure FF. Per-market share of sellers whose estimated propensity
# score falls inside the common-support range [overlap_low,
# overlap_high] (left panel), and the same coverage against the
# within-market FBA share, with marker colour encoding the
# within-market seller count (right panel). A linear reference line
# summarises the cross-market relationship.
# -----------------------------------------------------------------------
import statsmodels.api as sm, statsmodels.formula.api as smf

ps_formula_ff = ('fba_from_shipper ~ log1p_num_valutazioni + valutazioni_positive + '
                 'prezzo_totale_reconstructed + stelle + C(market_id)')
ps_ff = smf.glm(ps_formula_ff, data=df_final,
                family=sm.families.Binomial()).fit(maxiter=100, disp=0)
df_final['ps_ff'] = ps_ff.predict(df_final)
df_final['in_cs_ff'] = df_final['ps_ff'].between(overlap_low, overlap_high)

mkt_ff = df_final.groupby('market_order').agg(
    n_total=('in_cs_ff','count'),
    n_cs=('in_cs_ff','sum'),
    n_fba=('fba_from_shipper','sum'),
).reset_index()
mkt_ff['cs_share']  = mkt_ff['n_cs']/mkt_ff['n_total']
mkt_ff['fba_share'] = mkt_ff['n_fba']/mkt_ff['n_total']
overall_cs_ff = mkt_ff['cs_share'].mean()
print(f"CS coverage: mean={overall_cs_ff:.3f}, min={mkt_ff['cs_share'].min():.3f}, "
      f"max={mkt_ff['cs_share'].max():.3f}")

fig_ff, (ax_ff1, ax_ff2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax_ff1.scatter(mkt_ff['market_order'], mkt_ff['cs_share'],
               s=28, color=COLOR_FBA_FILL, edgecolor=COLOR_FBA_LINE, linewidth=0.7, zorder=3)
ax_ff1.plot(mkt_ff['market_order'], mkt_ff['cs_share'],
            color=COLOR_FBA_LINE, linewidth=0.7, alpha=0.5)
ax_ff1.axhline(overall_cs_ff, color='black', linestyle='--', linewidth=0.9)
ax_ff1.text(1, overall_cs_ff - 0.025, f'  mean = {overall_cs_ff:.1%}', fontsize=8.5, va='top')
ax_ff1.set_xlabel('Market index (chronological order, 62 markets)', fontsize=9)
ax_ff1.set_ylabel(f'Fraction within common support [{overlap_low:.2f},{overlap_high:.2f}]', fontsize=9)
ax_ff1.set_ylim(0,1); ax_ff1.set_title('Per-market common-support coverage', loc='left', fontsize=10)
ax_ff1.tick_params(direction='out', length=3)

sc_ff = ax_ff2.scatter(mkt_ff['fba_share'], mkt_ff['cs_share'],
                       c=mkt_ff['n_total'], cmap='YlOrRd', s=35,
                       edgecolor='black', linewidth=0.5, zorder=3)
ax_ff2.axhline(overall_cs_ff, color='black', linestyle='--', linewidth=0.8, alpha=0.6)
cbar_ff = plt.colorbar(sc_ff, ax=ax_ff2, shrink=0.8)
cbar_ff.set_label('Market size (n sellers)', fontsize=8)
cbar_ff.ax.tick_params(labelsize=7)
z_ff = np.polyfit(mkt_ff['fba_share'], mkt_ff['cs_share'], 1)
x_ff = np.linspace(mkt_ff['fba_share'].min(), mkt_ff['fba_share'].max(), 100)
ax_ff2.plot(x_ff, np.polyval(z_ff, x_ff), color='black', linewidth=1.0, linestyle=':', alpha=0.7)
ax_ff2.set_xlabel('FBA share within market', fontsize=9)
ax_ff2.set_ylabel('Common-support coverage', fontsize=9)
ax_ff2.set_title('CS coverage vs FBA share per market', loc='left', fontsize=10)
ax_ff2.tick_params(direction='out', length=3)

fig_ff.tight_layout()
fF_png = save_figure(fig_ff, 'fig_FF_propensity_overlap_by_market')
print('Saved', fF_png)


---

## Output manifest

The notebook produces 24 PDF figures (and matching PNGs for screen
inspection) under
`Amazon-BuyBox-Econometrics-Analysis/Datasets/Figures` on the mounted
Drive, or under the local staging directory if Drive is not mounted.
The PDF stems below are the targets of the `\includegraphics{}` calls
in `main.tex`.

| ID    | File stem                                       | LaTeX label                           |
|-------|-------------------------------------------------|----------------------------------------|
| F1    | `fig_F1_rank_pct_density`                       | `fig:rank-pct-density`                |
| F2    | `fig_F2_fba_share_by_rank_tier`                 | `fig:fba-share-by-rank-tier`          |
| F3    | `fig_F3_partial_regression_plot_fba_S4`         | `fig:fwl-fba-s4`                      |
| F4    | `fig_F4_residual_diagnostics_S4_D1`             | `fig:residual-diagnostics-s4-d1`      |
| F5    | `fig_F5_price_vs_rank_by_fba`                   | `fig:price-vs-rank-by-fba`            |
| F6    | `fig_F6_propensity_density_by_fba`              | `fig:propensity-density-by-fba`       |
| F7    | `fig_F7_non_fba_rank_composition_trim`          | `fig:nonfba-rank-composition-trim`    |
| F8    | `fig_F8_smd_before_after_trim`                  | `fig:smd-before-after-trim`           |
| F9    | `fig_F9_dropouts_distribution_by_focal_fba`     | `fig:dropouts-distribution-by-fba`    |
| F10   | `fig_F10_dynamic_means_by_focal_fba`            | `fig:dynamic-means-by-fba`            |
| F11   | `fig_F11_covariate_profile_by_fba`              | `fig:covariate-profile-by-fba`        |
| F12   | `fig_F12_panel_structure_timeline`              | `fig:panel-structure-timeline`        |
| F13a  | `fig_F13a_heatmap_price_stars`                  | `fig:rank-heatmap-price-stars`        |
| F13b  | `fig_F13b_heatmap_price_delivery`               | `fig:rank-heatmap-price-delivery`     |
| F13c  | `fig_F13c_heatmap_price_shipping`               | `fig:rank-heatmap-price-shipping`     |
| F14   | `fig_F14_market_structure_stability`            | `fig:market-structure-stability`      |
| F19   | `fig_F19_added_variable_plots_S4`               | `fig:avp-s4-covariates`               |
| F20   | `fig_F20_fba_prediction_model`                  | `fig:fba-prediction-model`            |
| FA    | `fig_FA_attenuation_decomposition_S1_S4`        | `fig:attenuation-decomposition`       |
| FB    | `fig_FB_within_market_demeaned_gap`             | `fig:within-market-demeaned-gap`      |
| FC    | `fig_FC_per_market_fba_gap`                     | `fig:per-market-fba-gap`              |
| FD    | `fig_FD_d1_placebo_test`                        | `fig:d1-placebo`                      |
| FE    | `fig_FE_seller_tenure_panel`                    | `fig:seller-tenure-by-fba`            |
| FF    | `fig_FF_propensity_overlap_by_market`           | `fig:propensity-coverage-by-market`   |


In [ ]:
# -----------------------------------------------------------------------------
# Summary listing of produced PNG figure files.
# (PDF counterparts are written alongside; the manifest cell above
# enumerates the 24 PDF stems.)
# -----------------------------------------------------------------------------
produced = sorted(FIGURES_DIR.glob('fig_F*.png'))
for p in produced:
    print(p.name, '-', p.stat().st_size, 'bytes')
print(f'Total: {len(produced)} PNG figures in {FIGURES_DIR}')
